# FADING Evaluation on FG-NET — Pipeline Suy diễn & Đánh giá Độc lập
Notebook đánh giá chuyên biệt trên tập dữ liệu **FG-NET** (Cross-age Re-identification đối chiếu ảnh thật ground truth).
Tự động chạy khép kín từ đầu đến cuối, hỗ trợ resume phiên chạy ngắt quãng, lưu báo cáo metrics và đóng gói kết quả.


In [ ]:
# ===== 1. KHAI BÁO TOÀN BỘ ĐƯỜNG DẪN & KIỂM TRA SẴN SÀNG (FG-NET) =====
import os
import sys
import datetime
import re
import zipfile
import torch

print("Các dataset có sẵn trong /kaggle/input:")
if os.path.exists("/kaggle/input"):
    for item in sorted(os.listdir("/kaggle/input")):
        print(f"  /kaggle/input/{item}")

# 1.1 Thư mục dữ liệu chính & thư mục xuất kết quả
DATA_DIR = "/kaggle/input/datasets/menonkk/nckh-2025-2026"
if not os.path.isdir(DATA_DIR):
    for alt in [
        "/kaggle/input/nckh-2025-2026",
        r"d:\Data\project\nckh\data",
        r"d:\Data\project\nckh",
    ]:
        if os.path.isdir(alt):
            DATA_DIR = alt
            break

OUTPUT_DIR = "/kaggle/working/FADING_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1.2 Checkpoint UNet chuyên biệt (512x512)
CKPT_DIR = DATA_DIR  # Trỏ vào dataset chứa checkpoint chuyên biệt UNet v2 AdamW
if not (os.path.isdir(CKPT_DIR) and any(f.endswith((".bin", ".safetensors")) for f in os.listdir(CKPT_DIR))):
    for alt_ckpt in [
        "/kaggle/working/FADING_output/checkpoints/specialized_unet_v2_adamw",
        "/kaggle/working/checkpoints/specialized_unet",
        r"d:\Data\project\nckh\checkpoints\specialized_unet",
    ]:
        if os.path.isdir(alt_ckpt):
            CKPT_DIR = alt_ckpt
            break

# 1.3 Thư mục ảnh FG-NET đã tiền xử lý offline
PREPROCESSED_FGNET_DIR = "/kaggle/input/fgnet-preprocessed-full/FGNET_preprocessed"
if not os.path.isdir(PREPROCESSED_FGNET_DIR):
    for alt_prep in [
        "/kaggle/working/FGNET_preprocessed",
        os.path.join(DATA_DIR, "FGNET_preprocessed"),
        os.path.join(DATA_DIR, "FGNET_preprocessed_full"),
        os.path.join(OUTPUT_DIR, "FGNET_preprocessed"),
        r"d:\Data\project\nckh\data\FGNET_preprocessed",
        r"d:\Data\project\nckh\data\FGNET_preprocessed_full",
    ]:
        if os.path.isdir(alt_prep):
            PREPROCESSED_FGNET_DIR = alt_prep
            break

# 1.4 Toàn bộ logic dò tìm / giải nén thư mục ảnh FG-NET gốc
def find_jpg_dir(base_dir):
    """Tìm thư mục đầu tiên có chứa ảnh .jpg/.jpeg (đệ quy qua mọi thư mục con)."""
    for root, dirs, files in os.walk(base_dir):
        if any(f.lower().endswith((".jpg", ".jpeg")) for f in files):
            return root
    return None

FGNET_SUBDIR_CANDIDATES = [
    os.path.join(DATA_DIR, "FGNET (1)"),
    os.path.join(DATA_DIR, "FGNET"),
    r"d:\Data\project\nckh\data\FGNET (1)",
    r"d:\Data\project\nckh\data\FGNET",
]
fgnet_base_dir = next((p for p in FGNET_SUBDIR_CANDIDATES if os.path.isdir(p)), None)
fgnet_images_dir = find_jpg_dir(fgnet_base_dir) if fgnet_base_dir else None

FGNET_EXTRACT_DIR = os.path.join(OUTPUT_DIR, "FGNET_extracted")
if fgnet_images_dir is None:
    FGNET_ZIP_CANDIDATES = [
        os.path.join(DATA_DIR, "FGNET (1).zip"),
        os.path.join(DATA_DIR, "FGNET.zip"),
        r"d:\Data\project\nckh\data\FGNET (1).zip",
    ]
    fgnet_zip_path = next((p for p in FGNET_ZIP_CANDIDATES if os.path.isfile(p)), None)
    if fgnet_zip_path is not None:
        print(f"Đang giải nén FG-NET từ {fgnet_zip_path} vào {FGNET_EXTRACT_DIR}...")
        os.makedirs(FGNET_EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(fgnet_zip_path, "r") as zf:
            zf.extractall(FGNET_EXTRACT_DIR)
        fgnet_images_dir = find_jpg_dir(FGNET_EXTRACT_DIR)

if fgnet_images_dir is not None:
    print(f"✅ Thư mục ảnh FG-NET xác định: {fgnet_images_dir}")
else:
    print("⚠️ Chưa tìm thấy ảnh FG-NET gốc ngay tại cell 1 (sẽ kiểm tra lại ở bước nạp dữ liệu).")

# 1.5 Cấu hình nguồn checkpoint MiVOLO
MIVOLO_DETECTOR_REPO = "iitolstykh/demo_yolov8_detector"
MIVOLO_DETECTOR_FILE = "yolov8x_person_face.pt"
MIVOLO_AGE_CKPT_SOURCES = [
    ("MuGeminorum/MiVOLO", "model/model_imdb_cross_person_4.22_99.46.pth.tar"),
    ("typorch/mivolo-imdb_cross_person-yolov8x_person_face", "model_imdb_cross_person_4.22_99.46.pth.tar"),
]
MIVOLO_LOCAL_DIR = os.path.join(DATA_DIR, "checkpoints/mivolo")
if not os.path.isdir(MIVOLO_LOCAL_DIR):
    MIVOLO_LOCAL_DIR = r"d:\Data\project\nckh\checkpoints\mivolo"

# 1.6 Dữ liệu ảnh demo Single Image
FFHQ_DIR = os.path.join(DATA_DIR, "ffhq_aging_150_samples/ffhq_aging_150_samples")
if not os.path.isdir(FFHQ_DIR):
    for alt_ffhq in [
        os.path.join(DATA_DIR, "ffhq_aging_150_samples"),
        r"d:\Data\project\nckh\ffhq_aging_150_samples\ffhq_aging_150_samples",
        r"d:\Data\project\nckh\ffhq_aging_150_samples",
    ]:
        if os.path.isdir(alt_ffhq):
            FFHQ_DIR = alt_ffhq
            break
LABELS_CSV = os.path.join(FFHQ_DIR, "sampled_labels.csv") if os.path.isdir(FFHQ_DIR) else ""
TEST_IMAGE_NAME = "01366.png"

# 1.7 Kiểm tra sẵn sàng hệ thống
print("=" * 70)
print("KIỂM TRA SẴN SÀNG TRƯỚC KHI CHẠY PIPELINE (FG-NET)")
print("=" * 70)
all_ok = True
def check(label, condition, detail=""):
    global all_ok
    status = "✅" if condition else "❌"
    if not condition:
        all_ok = False
    print(f"{status} {label}" + (f"  — {detail}" if detail else ""))

has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "không có"
check("GPU khả dụng", has_gpu, gpu_name if has_gpu else "Cần GPU CUDA để chạy pipeline")
check("DATA_DIR tồn tại", os.path.isdir(DATA_DIR), DATA_DIR)
check("Thư mục ảnh FG-NET tiền xử lý", os.path.isdir(PREPROCESSED_FGNET_DIR), PREPROCESSED_FGNET_DIR if os.path.isdir(PREPROCESSED_FGNET_DIR) else "Chưa tìm thấy -> Sẽ fallback ảnh gốc")
check("Thư mục ảnh FG-NET gốc khả dụng", fgnet_images_dir is not None and os.path.isdir(fgnet_images_dir), fgnet_images_dir)
if CKPT_DIR is not None and os.path.isdir(CKPT_DIR):
    check(f"Sử dụng checkpoint UNet tùy chỉnh tại '{CKPT_DIR}'", True, "Checkpoint đã nạp")
else:
    check("Sử dụng UNet SD 1.5 nguyên bản", True, "runwayml/stable-diffusion-v1-5")
print("=" * 70)
if all_ok:
    print("✅ TẤT CẢ ĐÃ SẴN SÀNG — có thể chạy tiếp các cell bên dưới.")
else:
    print("⚠️ CÒN MỤC CHƯA SẴN SÀNG — xem lại các cảnh báo phía trên.")
print("=" * 70)


## Ghi log toàn bộ session ra file — để gửi cho Claude đọc

Từ đây trở đi, MỌI dòng `print()` (kể cả log của các cell sau) sẽ vừa hiện trên
màn hình như bình thường, vừa được ghi thêm vào 1 file `.txt` trong OUTPUT_DIR — không
cần copy tay từng đoạn log nữa, chỉ cần gửi file này.

Dùng chế độ **append** (nối thêm, không ghi đè) — nếu phiên Kaggle bị ngắt rồi mở
lại, chạy lại đúng cell này, log cũ vẫn còn nguyên, log mới nối tiếp vào sau.

In [ ]:
import sys
import datetime

LOG_FILE_PATH = os.path.join(OUTPUT_DIR, "full_session_log.txt")

# SUA (lan 2): "isinstance(sys.stdout, TeeLogger)" KHONG dang tin - moi lan cell
# nay chay lai, Python tao ra 1 class TeeLogger MOI (du code giong het), nen
# isinstance luon that bai, cu the boc them 1 lop moi -> boc chong vo han lan,
# gay loi de quy AttributeError (thay ro trong traceback: isatty() goi lap 4 lan).
#
# Cach sua dung tin cay: "boc tach" theo kieu duck-typing (kiem tra co thuoc tinh
# .terminal hay khong, khong quan tam class nao), lan xuong TAN GOC stdout that su
# (khong con thuoc tinh .terminal nua) - bat ke da bi boc chong bao nhieu lop truoc do.
_true_stdout = sys.stdout
while hasattr(_true_stdout, "terminal"):
    _true_stdout = _true_stdout.terminal

class TeeLogger:
    """Ghi dong thoi ra man hinh (terminal) VA ra file - khong mat cai nao ca."""
    def __init__(self, filepath, mode="a"):
        self.terminal = _true_stdout   # LUON tro thang ve stdout GOC, khong bao gio boc chong
        self.log_file = open(filepath, mode, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def isatty(self):
        return self.terminal.isatty()

    def __getattr__(self, name):
        return getattr(self.terminal, name)

sys.stdout = TeeLogger(LOG_FILE_PATH, mode="a")

print(f"\n{'='*70}")
print(f"===== BẮT ĐẦU / TIẾP TỤC GHI LOG — {datetime.datetime.now()} =====")
print(f"===== File log: {LOG_FILE_PATH} =====")
print(f"{'='*70}\n")


## Cài đặt thư viện

Colab đã có sẵn `torch`, `numpy`, `pandas`, `pillow`, `opencv-python`, `tqdm` — chỉ cần cài thêm phần chưa có.

In [ ]:
# ===== 1. Cài đặt các thư viện cơ bản & mô hình =====
!pip install -q diffusers transformers accelerate bitsandbytes insightface onnxruntime-gpu huggingface_hub
!pip install -q "git+https://github.com/WildChlamydia/MiVOLO.git"
!pip install -q basicsr facexlib gfpgan

# ===== 2. Chuẩn bị CodeFormer cho pipeline tiền xử lý ảnh FG-NET =====
import os
import subprocess
import sys

CODEFORMER_DIR = "/kaggle/working/CodeFormer"
if not os.path.exists(CODEFORMER_DIR):
    print("Đang clone CodeFormer từ GitHub...")
    !git clone https://github.com/sczhou/CodeFormer.git {CODEFORMER_DIR}
    %cd {CODEFORMER_DIR}
    !pip install -q -r requirements.txt
    !python basicsr/setup.py develop
    %cd /kaggle/working
    
    # Tải weights facelib và CodeFormer
    print("Đang tải pretrained weights cho CodeFormer...")
    %cd {CODEFORMER_DIR}
    !python scripts/download_pretrained_models.py facelib
    !python scripts/download_pretrained_models.py CodeFormer
    %cd /kaggle/working
    print("✅ Đã tải xong toàn bộ weights của CodeFormer.")
else:
    print("Thư mục CodeFormer đã tồn tại.")

# ===== 3. Vá lỗi torchvision.transforms.functional_tensor trong basicsr =====
res = subprocess.run([sys.executable, "-m", "pip", "show", "basicsr"], capture_output=True, text=True)
site_packages_dir = None
for line in res.stdout.splitlines():
    if line.startswith("Location:"):
        site_packages_dir = line.split("Location:")[1].strip()
        break

files_to_patch = []
if site_packages_dir:
    files_to_patch.append(os.path.join(site_packages_dir, "basicsr", "data", "degradations.py"))
files_to_patch.append(os.path.join(CODEFORMER_DIR, "basicsr", "data", "degradations.py"))

patched_count = 0
for deg_file in files_to_patch:
    if os.path.exists(deg_file):
        with open(deg_file, "r", encoding="utf-8") as f:
            content = f.read()
        if "functional_tensor" in content:
            content = content.replace(
                "from torchvision.transforms.functional_tensor import",
                "from torchvision.transforms.functional import"
            )
            content = content.replace("functional_tensor", "functional")
            with open(deg_file, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"✅ Đã vá lỗi functional_tensor tại: {deg_file}")
            patched_count += 1
        else:
            print(f"ℹ️ File {deg_file} đã chuẩn (không chứa functional_tensor).")

print(f"\nHoàn tất chuẩn bị môi trường & CodeFormer. Số file đã vá: {patched_count}")


## Cấu hình pipeline (hyperparameters) — tương đương `configs/config.yaml`

In [ ]:
config = {
    "base_model": {
        "pretrained_model_name_or_path": "runwayml/stable-diffusion-v1-5",
    },
    "inversion": {
        "num_inference_steps": 50,
        "guidance_scale": 1.0,           # Đặt 1.0 cho DDIM Inversion thuần túy để bảo toàn 100% phông nền và áo sọc gốc
        "num_inner_steps": 10,           # 10 bước chuẩn FADING gốc (chống over-fitting)
        "early_stop_epsilon": 1e-5,      # Ngưỡng dừng chuẩn
        "image_size": 512,
    },
    "editing": {
        "guidance_scale": 4.0,           # Module Editing giữ 4.0 để sinh nếp nhăn già hóa
        "attention_control_ratio": 0.8,
        "image_size": 512,
    },
    "embedding": {
        "model_name": "buffalo_l",
        "ctx_id": -1,
        "det_size": (256, 256),
    },
    "paths": {
        "ffhq_dir": FFHQ_DIR,
        "labels_csv": LABELS_CSV,
        "specialized_unet_ckpt": CKPT_DIR,
        "output_dir": os.path.join(OUTPUT_DIR, "edited_images"),
    },
}
config


## Hàm tiện ích dùng chung & Prompt Helpers (tương đương `src/utils/prompts.py`)


In [ ]:
from typing import Dict
# Trung diem tung age_group trong sampled_labels.csv.
# Rieng nhom cuoi "70-120" LAY TAY = 80, khong dung trung diem toan hoc (se ra 95),
# vi paper FADING goc ghi ro: "For the oldest age group (70+), we translate to 80 years old".
AGE_GROUP_TO_AGE: Dict[str, int] = {
    "0-2": 1,
    "3-6": 4,
    "7-9": 8,
    "10-14": 12,
    "15-19": 17,
    "20-29": 24,
    "30-39": 34,
    "40-49": 44,
    "50-69": 59,
    "70-120": 80,
}
def age_group_to_age(age_group: str) -> int:
    """Quy doi 1 nhan age_group (vd "30-39") sang 1 con so tuoi dai dien (vd 34)."""
    return AGE_GROUP_TO_AGE[age_group]
def gender_to_word(gender: str, age: int) -> str:
    """Quy doi gender ("male"/"female") + tuoi sang tu mo ta gioi tinh dung trong prompt:
    woman/man cho nguoi lon (age >= 15), girl/boy neu age < 15."""
    is_female = gender.lower() == "female"
    if age < 15:
        return "girl" if is_female else "boy"
    return "woman" if is_female else "man"
def build_prompt_alpha(age: int, gender_word: str) -> str:
    """Build P_alpha = "photo of a {age} year old {gender_word}"."""
    return f"photo of a {age} year old {gender_word}"
def build_prompt_neutral(gender_word: str) -> str:
    """Build P_neutral = "photo of a {gender_word}" - prompt trung lap, khong chua tuoi."""
    return f"photo of a {gender_word}"
def build_prompt_tau(target_age: int, gender_word: str) -> str:
    """Build P_tau chuẩn FADING gốc (prompt đơn giản, không thêm Enhanced Prompts)."""
    return f"photo of a {target_age} year old {gender_word}"


## Module 2 — Null-text Inversion (tương đương `src/fading/inversion.py`)

In [ ]:
from typing import Dict as _Dict, Optional, List, Tuple
import os
import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.optim import Adam
from PIL import Image
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
NUM_DDIM_STEPS_DEFAULT = 50
GUIDANCE_SCALE_DEFAULT = 4.0
# Chi luu attention map cua layer co do phan giai <= 32x32 - giam VRAM, dung theo toi uu cua
# Prompt-to-Prompt goc (layer do phan giai thap mang nhieu ngu nghia hon).
MAX_ATTN_RESOLUTION = 32 * 32
class DualAttentionCapture:
    """
    Bắt đồng thời Self-Attention (attn1) và Cross-Attention (attn2)
    trong quá trình DDIM Inversion.
    """
    def __init__(self, unet: UNet2DConditionModel):
        self.unet = unet
        self._orig_processors = unet.attn_processors
        self.captured_self: _Dict[str, torch.Tensor] = {}
        self.captured_cross: _Dict[str, torch.Tensor] = {}
        self.enabled = False
    def _build_processor(self, name: str):
        capture = self
        is_cross = name.endswith("attn2.processor")
        class _CapturingProcessor:
            def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
                query = attn.to_q(hidden_states)
                context = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
                key = attn.to_k(context)
                value = attn.to_v(context)
                query = attn.head_to_batch_dim(query)
                key = attn.head_to_batch_dim(key)
                value = attn.head_to_batch_dim(value)
                attention_probs = attn.get_attention_scores(query, key, attention_mask)
                # Chỉ lưu attention map của các layer có số lượng spatial tokens <= 32x32
                if capture.enabled and attention_probs.shape[1] <= MAX_ATTN_RESOLUTION:
                    if is_cross:
                        capture.captured_cross[name] = attention_probs.detach().cpu()
                    else:
                        capture.captured_self[name] = attention_probs.detach().cpu()
                hidden_states = torch.bmm(attention_probs, value)
                hidden_states = attn.batch_to_head_dim(hidden_states)
                hidden_states = attn.to_out[0](hidden_states)
                hidden_states = attn.to_out[1](hidden_states)
                return hidden_states
        return _CapturingProcessor()
    def register(self) -> None:
        # Gắn processor vào toàn bộ các layer attention (cả attn1 và attn2)
        new_processors = {name: self._build_processor(name) for name in self.unet.attn_processors.keys()}
        self.unet.set_attn_processor(new_processors)
    def restore(self) -> None:
        self.unet.set_attn_processor(self._orig_processors)
    def capture_step(self, forward_fn):
        self.captured_self = {}
        self.captured_cross = {}
        self.enabled = True
        with torch.no_grad():
            result = forward_fn()
        self.enabled = False
        return dict(self.captured_self), dict(self.captured_cross), result
class NullTextInverter:
    def __init__(
        self,
        pretrained_model_name_or_path: str = "runwayml/stable-diffusion-v1-5",
        unet_checkpoint_dir: Optional[str] = None,
        device: str = "cuda",
        num_inference_steps: int = 50,         # Đưa về 50 bước chuẩn
        guidance_scale: float = 4.0,  # Mặc định 4.0 đồng bộ với config
        num_inner_steps: int = 10,
        early_stop_epsilon: float = 1e-5,
        image_size: int = 512,                 # Đưa lên 512x512
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.unet_checkpoint_dir = unet_checkpoint_dir
        self.device = device
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.num_inner_steps = num_inner_steps
        self.early_stop_epsilon = early_stop_epsilon
        self.image_size = image_size
        self.vae = None
        self.unet = None
        self.text_encoder = None
        self.tokenizer = None
        self.scheduler = None
    def _load_models(self) -> None:
        model_id = self.pretrained_model_name_or_path
        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)
        self.text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.float16)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        if self.unet_checkpoint_dir and os.path.isdir(self.unet_checkpoint_dir):
            self.unet = UNet2DConditionModel.from_pretrained(self.unet_checkpoint_dir, torch_dtype=torch.float16)
        else:
            self.unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=torch.float16)
        self.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
        self.scheduler.set_timesteps(self.num_inference_steps)
        self.vae.to(self.device).eval().requires_grad_(False)
        self.text_encoder.to(self.device).eval().requires_grad_(False)
        self.unet.to(self.device).eval().requires_grad_(False)
    def _load_image_latent(self, image_path: str) -> torch.Tensor:
        transform = transforms.Compose([
            transforms.Resize((self.image_size, self.image_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        image = Image.open(image_path).convert("RGB")
        image_t = transform(image).unsqueeze(0).to(self.device, dtype=torch.float16)
        with torch.no_grad():
            latent = self.vae.encode(image_t).latent_dist.mean * self.vae.config.scaling_factor
        return latent
    def _encode_text(self, prompt: str) -> torch.Tensor:
        tokens = self.tokenizer(
            [prompt],
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            return self.text_encoder(tokens.input_ids)[0]
    def _predict_noise(self, latent: torch.Tensor, t, embedding: torch.Tensor) -> torch.Tensor:
        return self.unet(latent, t, encoder_hidden_states=embedding).sample
    def _ddim_next_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        timestep, next_timestep = min(t - step, 999), t
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep] if timestep >= 0 else self.scheduler.final_alpha_cumprod
        alpha_prod_t_next = self.scheduler.alphas_cumprod[next_timestep]
        beta_prod_t = 1 - alpha_prod_t
        next_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        next_sample_direction = (1 - alpha_prod_t_next)**0.5 * noise_pred
        return alpha_prod_t_next**0.5 * next_original_sample + next_sample_direction
    def _ddim_prev_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        prev_timestep = t - step
        alpha_prod_t = self.scheduler.alphas_cumprod[t]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        pred_sample_direction = (1 - alpha_prod_t_prev)**0.5 * noise_pred
        return alpha_prod_t_prev**0.5 * pred_original_sample + pred_sample_direction
    def _ddim_inversion(self, z0: torch.Tensor, cond_embedding: torch.Tensor) -> List[torch.Tensor]:
        latent = z0.clone().detach()
        pivot_latents = [latent]
        timesteps = self.scheduler.timesteps
        for i in range(self.num_inference_steps):
            t = timesteps[len(timesteps) - i - 1]
            with torch.no_grad():
                noise_pred = self._predict_noise(latent, t, cond_embedding)
                latent = self._ddim_next_step(noise_pred, t, latent)
            pivot_latents.append(latent)
        return pivot_latents
    def _null_text_optimization(
        self,
        pivot_latents: List[torch.Tensor],
        uncond_embedding: torch.Tensor,
        cond_embedding: torch.Tensor,
        attn_capture: DualAttentionCapture,
    ):
        uncond_embeddings = uncond_embedding.clone()
        null_embeddings_list: List[torch.Tensor] = []
        self_attention_maps: _Dict[int, _Dict[str, torch.Tensor]] = {}
        cross_attention_maps: _Dict[int, _Dict[str, torch.Tensor]] = {}
        latent_cur = pivot_latents[-1]
        timesteps = self.scheduler.timesteps
        for i in range(self.num_inference_steps):
            uncond_embeddings = uncond_embeddings.clone().detach().float().requires_grad_(True)
            
            # Learning rate phân rã tuyến tính nhẹ
            lr_scale = 1.0 if i < 25 else max(0.4, 1.0 - (i - 25) / 35.0)
            optimizer = Adam([uncond_embeddings], lr=1e-2 * lr_scale)
            
            latent_prev = pivot_latents[len(pivot_latents) - i - 2]
            t = timesteps[i]
            
            with torch.no_grad():
                noise_pred_cond = self._predict_noise(latent_cur, t, cond_embedding)
                
            for inner_step in range(self.num_inner_steps):
                noise_pred_uncond = self._predict_noise(latent_cur, t, uncond_embeddings.half())
                noise_pred = noise_pred_uncond + self.guidance_scale * (noise_pred_cond - noise_pred_uncond)
                latent_prev_rec = self._ddim_prev_step(noise_pred, t, latent_cur)
                loss = F.mse_loss(latent_prev_rec, latent_prev)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                if loss.item() < self.early_stop_epsilon:
                    break
            null_embeddings_list.append(uncond_embeddings[:1].detach().half())
            with torch.no_grad():
                noise_pred_uncond_final = self._predict_noise(latent_cur, t, uncond_embeddings.half())
            # Bắt đồng thời cả Self và Cross Attention maps tại bước CFG thực
            s_maps, c_maps, noise_pred_cond_final = attn_capture.capture_step(
                lambda: self._predict_noise(latent_cur, t, cond_embedding)
            )
            self_attention_maps[int(t)] = s_maps
            cross_attention_maps[int(t)] = c_maps
            with torch.no_grad():
                noise_pred = noise_pred_uncond_final + self.guidance_scale * (
                    noise_pred_cond_final - noise_pred_uncond_final
                )
                latent_cur = self._ddim_prev_step(noise_pred, t, latent_cur)
            print(f"[NullTextInverter] t={int(t)} ({i + 1}/{self.num_inference_steps}) loss={loss.item():.6f}")
        return null_embeddings_list, (self_attention_maps, cross_attention_maps)
    def invert(self, image_path: str, initial_age: int, gender_word: str):
        if self.unet is None:
            self._load_models()
        p_alpha = build_prompt_alpha(initial_age, gender_word)
        uncond_embedding = self._encode_text("")
        cond_embedding = self._encode_text(p_alpha)
        z0 = self._load_image_latent(image_path)
        pivot_latents = self._ddim_inversion(z0, cond_embedding)
        # --- TỐI ƯU HÓA CHO TRẺ EM / TODDLER (initial_age < 10) ---
        # Trẻ em có cấu trúc xương mặt thay đổi mạnh, cần tăng số bước inner_steps 
        # trong Null-text optimization để bám sát đặc trưng không gian tốt hơn.
        original_inner_steps = self.num_inner_steps
        if initial_age < 10:
            self.num_inner_steps = max(self.num_inner_steps, 20)  # Tăng lên 20 bước cho trẻ nhỏ
            print(f"[NullTextInverter] Phát hiện độ tuổi trẻ em ({initial_age} tuổi) -> Tăng num_inner_steps lên {self.num_inner_steps} để tối ưu cấu trúc xương mặt.")
        else:
            print(f"[NullTextInverter] Độ tuổi người lớn/thiếu niên ({initial_age} tuổi) -> Dùng num_inner_steps chuẩn: {self.num_inner_steps}")
        attn_capture = DualAttentionCapture(self.unet)
        attn_capture.register()
        try:
            null_embeddings_list, (self_maps, cross_maps) = self._null_text_optimization(
                pivot_latents, uncond_embedding, cond_embedding, attn_capture
            )
        finally:
            attn_capture.restore()
            # Khôi phục lại giá trị gốc sau khi chạy xong
            self.num_inner_steps = original_inner_steps
        z_T = pivot_latents[-1]
        return z_T, null_embeddings_list, (self_maps, cross_maps)


## Module 3 — Editing (tương đương `src/fading/editing.py`)

In [ ]:
from typing import Dict as _Dict, Optional, List, Tuple
import os
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer


# SUA: Thêm hàm get_word_inds phục vụ cơ chế LocalBlend (Prompt-to-Prompt - Hertz et al.)
def get_word_inds(prompt: str, word: str, tokenizer) -> np.ndarray:
    """
    Tìm vị trí token của từ `word` trong `prompt` đã tokenize bởi CLIPTokenizer.
    Dùng để định vị token chủ thể chung (như "person", "woman", "man") giữa prompt gốc và prompt đích.
    Trả về mảng 1D các index token.
    """
    word_clean = word.strip().lower()
    tokens = tokenizer.encode(prompt)
    inds = []
    for idx, token_id in enumerate(tokens):
        tok_str = tokenizer.decode([token_id]).strip().lower().replace("</w>", "").strip(",.!?\"'")
        if tok_str and (tok_str == word_clean or word_clean in tok_str):
            inds.append(idx)
    if not inds:
        # Fallback nếu từ bị tách nhỏ qua subwords
        for idx, token_id in enumerate(tokens):
            tok_str = tokenizer.decode([token_id]).strip().lower()
            if word_clean in tok_str or (len(tok_str) > 2 and tok_str in word_clean):
                inds.append(idx)
    if not inds:
        # Fallback an toàn cuối cùng: token thứ 4 (thường là danh từ sau "photo of a")
        inds = [4]
    return np.array(inds, dtype=int)


# SUA: Triển khai công thức LocalBlend từ Prompt-to-Prompt gốc (pipeline_prompt2prompt.py)
def local_blend(
    recon_latent: torch.Tensor,
    edit_latent: torch.Tensor,
    cross_attn_maps_recon: List[torch.Tensor],
    cross_attn_maps_edit: List[torch.Tensor],
    word_inds_recon: np.ndarray,
    word_inds_edit: np.ndarray,
    threshold: float = 0.3,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Cơ chế LocalBlend (Hertz et al. - Prompt-to-Prompt):
    Chỉ cho phép thay đổi ở ĐÚNG vùng chứa chủ thể (face/person), GIỮ NGUYÊN latent gốc
    ở mọi vùng khác (nền, tóc, quần áo) tại MỖI bước denoising (không chỉ hậu xử lý 1 lần).

    Công thức gốc:
        maps = torch.cat([cross_attn_maps_recon, cross_attn_maps_edit], dim=0)
        maps = (maps * alpha_layers).sum(-1).mean(...)
        mask = F.max_pool2d(maps, 3, 1, padding=1)
        mask = F.interpolate(mask, size=recon_latent.shape[2:])
        mask = mask / mask.max(...)
        mask = mask.gt(threshold)
        mask = (mask[:1] | mask[1:]).to(dtype=edit_latent.dtype)
        return recon_latent + mask * (edit_latent - recon_latent)
    """
    if len(cross_attn_maps_recon) == 0 or len(cross_attn_maps_edit) == 0:
        return edit_latent, torch.ones_like(edit_latent[:, :1])

    if len(word_inds_recon) == 0:
        word_inds_recon = np.array([4])
    if len(word_inds_edit) == 0:
        word_inds_edit = np.array([4])

    # 1. Trích xuất cross-attention maps của các token chủ thể cho recon và edit
    # Mỗi map có shape [num_heads, 256, 77] (với 256 = 16x16)
    recon_maps = []
    for m in cross_attn_maps_recon:
        sub_m = m[:, :, word_inds_recon].mean(dim=-1).reshape(-1, 16, 16)
        recon_maps.append(sub_m)

    edit_maps = []
    for m in cross_attn_maps_edit:
        sub_m = m[:, :, word_inds_edit].mean(dim=-1).reshape(-1, 16, 16)
        edit_maps.append(sub_m)

    # 2. Gom các layer và tính trung bình theo heads và layers -> [1, 1, 16, 16]
    recon_avg = torch.cat(recon_maps, dim=0).mean(dim=0, keepdim=True).unsqueeze(0)
    edit_avg = torch.cat(edit_maps, dim=0).mean(dim=0, keepdim=True).unsqueeze(0)

    # 3. Ghép recon và edit: shape [2, 1, 16, 16]
    maps = torch.cat([recon_avg, edit_avg], dim=0)

    # 4. Max-pool 3x3 để mở rộng bao phủ biên chủ thể
    maps = F.max_pool2d(maps, kernel_size=3, stride=1, padding=1)

    # 5. Nội suy lên kích thước latent (64x64)
    maps = F.interpolate(maps, size=recon_latent.shape[2:], mode="bilinear", align_corners=False)

    # 6. Chuẩn hóa về [0, 1] cho mỗi map
    max_val = maps.flatten(2).max(dim=-1)[0].unsqueeze(-1).unsqueeze(-1).clamp(min=1e-8)
    norm_maps = maps / max_val

    # 7. Nhị phân hóa với ngưỡng threshold
    mask = norm_maps.gt(threshold)

    # 8. Hợp nhất: vùng chủ thể ở recon HOẶC ở edit -> shape [1, 1, H, W]
    # Ép kiểu mask theo đúng dtype của edit_latent (float16/float32) để tránh lỗi lệch dtype với UNet
    mask = (mask[:1] | mask[1:]).to(dtype=edit_latent.dtype)

    # 9. Pha trộn latent: recon_latent + mask * (edit_latent - recon_latent)
    blended = recon_latent + mask * (edit_latent - recon_latent)
    return blended, mask


class DualAttentionInjector:
    """
    Tiêm Self-Attention (attn1) để khóa hình học khuôn mặt,
    và tiêm Cross-Attention (attn2) có chọn lọc để hòa trộn tuổi tác mà không làm méo mặt.
    Bổ sung khả năng bắt live cross-attention maps của các layer 16x16 phục vụ LocalBlend.
    """
    def __init__(
        self,
        unet: UNet2DConditionModel,
        self_maps: _Dict[int, _Dict[str, torch.Tensor]],
        cross_maps: _Dict[int, _Dict[str, torch.Tensor]],
        attention_control_ratio: float,
        num_inference_steps: int,
    ):
        self.unet = unet
        self._orig_processors = unet.attn_processors
        self.self_maps = self_maps
        self.cross_maps = cross_maps
        self.attention_control_ratio = attention_control_ratio
        self.num_inference_steps = num_inference_steps
        self.enabled = False
        self.current_t: Optional[int] = None
        # SUA: Thêm trạng thái bắt live cross-attention cho LocalBlend
        self.capture_mode: Optional[str] = None  # "recon", "edit", hoặc None
        self.captured_cross: _Dict[str, List[torch.Tensor]] = {"recon": [], "edit": []}

    def _build_processor(self, name: str):
        injector = self
        is_cross = name.endswith("attn2.processor")
        class _InjectingProcessor:
            def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
                query = attn.to_q(hidden_states)
                context = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
                key = attn.to_k(context)
                value = attn.to_v(context)  # Value vector luôn tính từ prompt mới P_tau
                query = attn.head_to_batch_dim(query)
                key = attn.head_to_batch_dim(key)
                value = attn.head_to_batch_dim(value)
                attention_probs = attn.get_attention_scores(query, key, attention_mask)

                # SUA: bắt attention TRƯỚC khi bị ghi đè, đảm bảo LocalBlend dùng dữ liệu sống
                if injector.capture_mode is not None and is_cross and attention_probs.shape[1] == 256:
                    injector.captured_cross[injector.capture_mode].append(attention_probs.detach())

                if injector.enabled:
                    if not is_cross:
                        # 1. Khóa hình học khuôn mặt bằng Self-Attention (attn1)
                        ref_self = injector.self_maps.get(injector.current_t, {}).get(name)
                        if ref_self is not None:
                            attention_probs = ref_self.to(device=value.device, dtype=value.dtype)
                    else:
                        # 2. Tiêm Cross-Attention (attn2):
                        # Giữ các token chung: "<start>", "photo", "of", "a" (index 0..3)
                        # và các token đệm/EOS phía sau (index >= 7).
                        # Thả tự do các token tuổi (index 4..6: "{age}", "year", "old") để nếp nhăn sinh tự nhiên.
                        ref_cross = injector.cross_maps.get(injector.current_t, {}).get(name)
                        if ref_cross is not None:
                            ref_cross = ref_cross.to(device=value.device, dtype=value.dtype)
                            if attention_probs.shape[-1] == ref_cross.shape[-1]:
                                attention_probs[:, :, :4] = ref_cross[:, :, :4]
                                attention_probs[:, :, 7:] = ref_cross[:, :, 7:]

                hidden_states = torch.bmm(attention_probs, value)
                hidden_states = attn.batch_to_head_dim(hidden_states)
                hidden_states = attn.to_out[0](hidden_states)
                hidden_states = attn.to_out[1](hidden_states)
                return hidden_states
        return _InjectingProcessor()

    def register(self) -> None:
        # Gắn processor vào toàn bộ các layer attention (cả attn1 và attn2)
        new_processors = {name: self._build_processor(name) for name in self.unet.attn_processors.keys()}
        self.unet.set_attn_processor(new_processors)

    def restore(self) -> None:
        self.unet.set_attn_processor(self._orig_processors)

    def inject_step(self, step_index: int, t, forward_fn):
        self.current_t = int(t)
        self.enabled = step_index < (self.attention_control_ratio * self.num_inference_steps)
        with torch.no_grad():
            result = forward_fn()
        self.enabled = False
        return result


class Editor:
    def __init__(
        self,
        pretrained_model_name_or_path: str = "runwayml/stable-diffusion-v1-5",
        unet_checkpoint_dir: Optional[str] = None,
        device: str = "cuda",
        num_inference_steps: int = 50,
        guidance_scale: float = 4.0,  # Mặc định 4.0 đồng bộ với config
        attention_control_ratio: float = 0.8,
        image_size: int = 512,
        use_local_blend: bool = True,                # Bật lại LocalBlend mặc định (= True)
        local_blend_threshold: float = 0.3,          # SUA: Ngưỡng binarize 0.3 chuẩn Hertz et al.
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.unet_checkpoint_dir = unet_checkpoint_dir
        self.device = device
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.attention_control_ratio = attention_control_ratio
        self.image_size = image_size
        self.use_local_blend = use_local_blend
        self.local_blend_threshold = local_blend_threshold
        self.vae = None
        self.unet = None
        self.text_encoder = None
        self.tokenizer = None
        self.scheduler = None

    def _load_models(self) -> None:
        model_id = self.pretrained_model_name_or_path
        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)
        self.text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.float16)
        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        if self.unet_checkpoint_dir and os.path.isdir(self.unet_checkpoint_dir):
            self.unet = UNet2DConditionModel.from_pretrained(self.unet_checkpoint_dir, torch_dtype=torch.float16)
        else:
            self.unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=torch.float16)
        self.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
        self.scheduler.set_timesteps(self.num_inference_steps)
        self.vae.to(self.device).eval().requires_grad_(False)
        self.text_encoder.to(self.device).eval().requires_grad_(False)
        self.unet.to(self.device).eval().requires_grad_(False)

    def _encode_text(self, prompt: str) -> torch.Tensor:
        tokens = self.tokenizer(
            [prompt],
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            return self.text_encoder(tokens.input_ids)[0]

    def _predict_noise(self, latent: torch.Tensor, t, embedding: torch.Tensor) -> torch.Tensor:
        return self.unet(latent, t, encoder_hidden_states=embedding).sample

    def _ddim_prev_step(self, noise_pred: torch.Tensor, t: int, sample: torch.Tensor) -> torch.Tensor:
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        prev_timestep = t - step
        alpha_prod_t = self.scheduler.alphas_cumprod[t]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_original_sample = (sample - beta_prod_t**0.5 * noise_pred) / alpha_prod_t**0.5
        pred_sample_direction = (1 - alpha_prod_t_prev)**0.5 * noise_pred
        return alpha_prod_t_prev**0.5 * pred_original_sample + pred_sample_direction

    def _decode_latent_to_image(self, latent: torch.Tensor) -> Image.Image:
        if self.vae is None:
            self._load_models()
        with torch.no_grad():
            latent = latent / self.vae.config.scaling_factor
            self.vae.to(dtype=torch.float32)
            image = self.vae.decode(latent.float()).sample
            self.vae.to(dtype=torch.float16)
        image = (image / 2 + 0.5).clamp(0, 1)
        image_np = (image[0].permute(1, 2, 0).float().cpu().numpy() * 255).round().astype(np.uint8)
        return Image.fromarray(image_np)

    def _validate_timesteps_available(self, attention_maps, timesteps, num_injection_steps: int) -> None:
        missing = [int(t) for t in timesteps[:num_injection_steps] if int(t) not in attention_maps]
        if missing:
            raise ValueError(f"attention_maps thieu timestep {missing}. Kiểm tra num_inference_steps giữa Inverter và Editor.")
    def reconstruct(
        self,
        z_T: torch.Tensor,
        null_embeddings: List[torch.Tensor],
        initial_age: int,
        gender_word: str,
        guidance_scale: Optional[float] = 1.0,
    ) -> Image.Image:
        """
        Tái tạo lại ảnh ban đầu từ z_T và null_embeddings.
        Để kiểm tra độ trung thực (Sanity-Check) mà không bị CFG bóp méo thành tranh vẽ,
        sử dụng chính trajectory đảo ngược chuẩn xác của DDIM (mặc định guidance_scale=1.0 theo ODE gốc).
        """
        if self.unet is None or self.vae is None:
            self._load_models()
        g_scale = 1.0 if guidance_scale is None else guidance_scale
        p_alpha = build_prompt_alpha(initial_age, gender_word)
        cond_embedding = self._encode_text(p_alpha)
        latent = z_T.clone()
        timesteps = self.scheduler.timesteps
        
        with torch.no_grad():
            for i in range(self.num_inference_steps):
                t = timesteps[i]
                null_t = null_embeddings[i]
                
                # Dự đoán nhiễu với null-text optimization
                noise_uncond = self._predict_noise(latent, t, null_t)
                noise_cond = self._predict_noise(latent, t, cond_embedding)
                
                # Áp dụng guidance_scale đồng bộ với quá trình Inversion (1.0 theo chuẩn ODE vi phân)
                noise_pred = noise_uncond + g_scale * (noise_cond - noise_uncond)
                latent = self._ddim_prev_step(noise_pred, t, latent)
                
        return self._decode_latent_to_image(latent)

    def edit(
        self,
        z_T: torch.Tensor,
        null_embeddings: List[torch.Tensor],
        attention_maps: Tuple[_Dict, _Dict],
        target_ages: List[int],
        gender_word: str,
        output_dir: str,
        original_image_path: Optional[str] = None,
        embedder: Optional[object] = None,
        use_local_blend: bool = True,                # Bật lại LocalBlend mặc định (= True)
        local_blend_threshold: Optional[float] = None,# SUA: Ngưỡng binarize cho LocalBlend
        initial_age: Optional[int] = None,           # SUA: Độ tuổi gốc để tạo prompt reconstruction
    ) -> _Dict[int, str]:
        if self.unet is None:
            self._load_models()
        self_maps, cross_maps = attention_maps
        os.makedirs(output_dir, exist_ok=True)
        timesteps = self.scheduler.timesteps
        num_injection_steps = int(self.attention_control_ratio * self.num_inference_steps)
        self._validate_timesteps_available(self_maps, timesteps, num_injection_steps)

        # SUA: Xác định cờ use_local_blend và threshold
        do_local_blend = self.use_local_blend if use_local_blend is None else use_local_blend
        lb_threshold = self.local_blend_threshold if local_blend_threshold is None else local_blend_threshold

        results: _Dict[int, str] = {}
        for target_age in target_ages:
            p_tau = build_prompt_tau(target_age, gender_word)
            cond_embedding = self._encode_text(p_tau)
            latent = z_T.clone()

            # SUA: Chuẩn bị latent reconstruction và prompt neo cho LocalBlend
            if do_local_blend:
                recon_latent = z_T.clone()
                p_alpha = build_prompt_alpha(initial_age, gender_word) if initial_age is not None else f"photo of a {gender_word}"
                cond_embedding_recon = self._encode_text(p_alpha)
                word_inds_recon = get_word_inds(p_alpha, gender_word, self.tokenizer)
                word_inds_edit = get_word_inds(p_tau, gender_word, self.tokenizer)
                print(f"[Editor LocalBlend] Khởi chạy song song recon_latent | p_alpha='{p_alpha}' | p_tau='{p_tau[:35]}...' | threshold={lb_threshold}")

            # --- CỐ ĐỊNH CHUẨN ATTENTION CONTROL RATIO ---
            dynamic_ratio = self.attention_control_ratio  # Cố định = 0.8 theo đúng ablation study
            print(f"[Editor] target_age={target_age} -> dùng fixed attention_ratio={dynamic_ratio}")

            injector = DualAttentionInjector(
                self.unet, self_maps, cross_maps, dynamic_ratio, self.num_inference_steps
            )
            injector.register()
            try:
                for i in range(self.num_inference_steps):
                    t = timesteps[i]
                    null_t = null_embeddings[i]

                    # --- NẾU BẬT LOCALBLEND: Chạy bước denoising cho reconstruction latent song song ---
                    if do_local_blend:
                        injector.captured_cross["recon"].clear()
                        injector.captured_cross["edit"].clear()

                        with torch.no_grad():
                            noise_uncond_recon = self._predict_noise(recon_latent, t, null_t)

                        # Bật capture_mode="recon" cho conditional forward pass
                        injector.capture_mode = "recon"
                        injector.enabled = False  # Không tiêm attention vào recon pass
                        with torch.no_grad():
                            noise_cond_recon = self._predict_noise(recon_latent, t, cond_embedding_recon)
                        injector.capture_mode = None

                        with torch.no_grad():
                            noise_pred_recon = noise_uncond_recon + self.guidance_scale * (noise_cond_recon - noise_uncond_recon)
                            recon_latent = self._ddim_prev_step(noise_pred_recon, t, recon_latent)

                    # --- Denoise edit latent (tiêm attention bình thường) ---
                    with torch.no_grad():
                        noise_uncond = self._predict_noise(latent, t, null_t)

                    # Bật capture_mode="edit" cho conditional forward pass của edit
                    if do_local_blend:
                        injector.capture_mode = "edit"

                    noise_cond = injector.inject_step(
                        i, t, lambda: self._predict_noise(latent, t, cond_embedding)
                    )

                    if do_local_blend:
                        injector.capture_mode = None

                    with torch.no_grad():
                        noise_pred = noise_uncond + self.guidance_scale * (noise_cond - noise_uncond)
                        latent = self._ddim_prev_step(noise_pred, t, latent)

                        # SUA: Áp dụng LocalBlend TRỘN 2 latent ngay sau ddim_prev_step ở mỗi bước i
                        if do_local_blend:
                            if len(injector.captured_cross["recon"]) > 0 and len(injector.captured_cross["edit"]) > 0:
                                latent, _ = local_blend(
                                    recon_latent=recon_latent,
                                    edit_latent=latent,
                                    cross_attn_maps_recon=injector.captured_cross["recon"],
                                    cross_attn_maps_edit=injector.captured_cross["edit"],
                                    word_inds_recon=word_inds_recon,
                                    word_inds_edit=word_inds_edit,
                                    threshold=lb_threshold,
                                )
                            injector.captured_cross["recon"].clear()
                            injector.captured_cross["edit"].clear()

                        if torch.isnan(latent).any():
                            raise RuntimeError(f"[Editor] NaN xuat hien tai step i={i}, t={int(t)}, target_age={target_age}.")
            finally:
                injector.restore()

            image = self._decode_latent_to_image(latent)

            # --- TẮT MASK-BASED BLENDING BẰNG ELLIPSE (Khôi phục pipeline FADING thuần túy) ---


            # Bỏ bước apply_mask_blending để mô hình Diffusion tự do render toàn bộ khuôn mặt,


            # tóc và cổ tương ứng với từng độ tuổi thay vì cắt dán viền ellipse đè lên cổ/áo ảnh gốc.


            # if original_image_path and os.path.exists(original_image_path) and embedder is not None:


            #     image = apply_mask_blending(original_image_path, image, embedder)



            path = os.path.join(output_dir, f"age_{target_age}.png")
            image.save(path)
            results[target_age] = path
            print(f"[Editor] target_age={target_age} -> {path}")
        return results


## Module 4 — InsightFace Embedding (tương đương `src/search/embedding.py`)

In [ ]:
import glob

import cv2

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")


class FaceEmbedder:
    """Boc insightface.app.FaceAnalysis: trich xuat embedding 512-chieu cho 1 anh, va build
    gallery embedding cho toan bo anh trong 1 folder."""

    def __init__(
        self,
        model_name: str = "buffalo_l",
        ctx_id: int = -1,
        det_size: Tuple[int, int] = (256, 256),
    ):
        """Luu config (ctx_id: -1=CPU mac dinh de tranh tranh chap VRAM voi cac module
        diffusion). Model CHUA duoc load o day, se load lazily trong _load_model()."""
        self.model_name = model_name
        self.ctx_id = ctx_id
        self.det_size = det_size
        self.app = None

    def _load_model(self) -> None:
        """Load FaceAnalysis(buffalo_l) va prepare() theo dung ctx_id/det_size."""
        from insightface.app import FaceAnalysis

        self.app = FaceAnalysis(name=self.model_name)
        self.app.prepare(ctx_id=self.ctx_id, det_size=self.det_size)

    def embed(self, image_path: str) -> np.ndarray:
        """Doc anh bang cv2.imread (BGR - dung chuan insightface, KHONG dung PIL vi PIL doc
        RGB se cho embedding sai lech ma khong bao loi), tra ve normed_embedding (512-dim).
        Neu khong detect duoc mat, raise ValueError ro rang (khong tra ve None/vector rong)."""
        if self.app is None:
            self._load_model()

        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Khong doc duoc anh: {image_path}")

        faces = self.app.get(img)
        if len(faces) == 0:
            raise ValueError(f"Khong phat hien duoc khuon mat nao trong anh: {image_path}")

        return faces[0].normed_embedding

    def detect_faces(self, image_path: str) -> list:
        """SUA (moi them): tra ve list cac mat detect duoc (bbox, kps 5 diem, det_score).
        kps dung de align_to_ffhq() - QUAN TRONG: bi thieu tu dau, khien vong lap FG-NET
        chua tung align anh, dan den ket qua co the thap hon nang luc that cua he thong."""
        if self.app is None:
            self._load_model()
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Khong doc duoc anh: {image_path}")
        faces = self.app.get(img)
        # SUA: them "gender" (0=nu, 1=nam, InsightFace tra ve san, khong ton gi them)
        # - dung cho Enhanced Prompts (EP), da ghi chu la quan trong nhung CHUA tung
        # duoc ap dung trong vong lap danh gia FG-NET (luon dung "person" co dinh).
        return [{"bbox": f.bbox, "kps": f.kps, "det_score": f.det_score,
                  "gender": int(f.gender)} for f in faces]

    def build_gallery(self, folder_path: str) -> Tuple[List[np.ndarray], List[str], List[str]]:
        """Chay embedding cho toan bo anh trong folder_path. Anh loi KHONG lam dung ca ham,
        nhung cung KHONG am tham bo qua: gom vao failed_files. Tra ve
        (embeddings, identity_labels, failed_files)."""
        image_paths = sorted(
            f
            for f in glob.glob(os.path.join(folder_path, "*"))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )

        embeddings: List[np.ndarray] = []
        identity_labels: List[str] = []
        failed_files: List[str] = []

        for image_path in image_paths:
            try:
                embedding = self.embed(image_path)
            except ValueError as e:
                print(f"[FaceEmbedder] BO QUA anh loi: {e}")
                failed_files.append(image_path)
                continue
            embeddings.append(embedding)
            identity_labels.append(os.path.splitext(os.path.basename(image_path))[0])

        if failed_files:
            print(
                f"[FaceEmbedder] Gallery '{folder_path}' thieu {len(failed_files)}/"
                f"{len(image_paths)} identity do loi doc/detect anh."
            )

        return embeddings, identity_labels, failed_files

## Tiền xử lý & Hậu xử lý khuôn mặt (Preprocessing, Face Alignment & Mask-based Blending)

1. **Pipeline Tiền xử lý (Được kiểm chứng từ `test_preprocessing_fgnet.ipynb`):**
   - **`apply_adaptive_padding`:** Dùng `cv2.BORDER_REPLICATE` thay vì reflect để triệt tiêu hoa văn hình thoi khi mặt sát mép ảnh.
   - **`apply_white_balance`:** Thuật toán **Shades of Gray (Minkowski p-norm, $p=6$, Finlayson & Trezzi 2004)** kèm dynamic alpha-blending ($\Delta_{\max} > 35.0$), khử vệt xanh trán ở specular highlight và bảo toàn sắc da ấm tự nhiên.
   - **`run_codeformer`:** Phục hồi chi tiết khuôn mặt với fidelity weight duy nhất $w=0.7$ (Fidelity Focus) qua CodeFormer CLI `--face_upsample`.
2. **Face Alignment:** Căn chỉnh khuôn mặt về chuẩn 512x512 dựa trên landmarks InsightFace (chuẩn NVIDIA FFHQ).
3. **Mask-based Blending:** Hậu xử lý pha trộn vùng mặt đã sinh vào ảnh gốc bằng soft mask để bảo toàn phông nền, tóc và áo quần gốc.


In [ ]:
from PIL import Image
import numpy as np
import cv2
import os
import subprocess
import sys
import shutil
from typing import Tuple, List, Dict, Optional
from scipy.ndimage import gaussian_filter

# ==============================================================================
# 1. PIPELINE TIỀN XỬ LÝ (ADAPTIVE PADDING + SHADES OF GRAY WB + CODEFORMER w=0.7)
# ==============================================================================

def apply_adaptive_padding(
    image_rgb: np.ndarray, 
    face_occupancy_thresh: float = 0.85, 
    pad_ratio: float = 0.20,
    border_mode: str = "replicate",
    embedder = None
) -> Tuple[np.ndarray, bool]:
    """
    Bước 1: Adaptive Padding với cv2.BORDER_REPLICATE.
    Nếu khuôn mặt chiếm >= 85% chiều dài/rộng ảnh gốc hoặc sát mép (< 5% biên),
    thêm padding lặp mép 20% mỗi cạnh. Dùng BORDER_REPLICATE để tránh hoa văn hình thoi do phản chiếu mép mặt.
    """
    H, W = image_rgb.shape[:2]
    faces = []
    if embedder is not None and hasattr(embedder, 'detect_faces'):
        try:
            faces = embedder.detect_faces(image_rgb)
        except Exception:
            pass

    if len(faces) == 0:
        try:
            cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
            face_cascade = cv2.CascadeClassifier(cascade_path)
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            h_faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))
            for (x, y, w, h) in h_faces:
                faces.append({"bbox": [x, y, x + w, y + h]})
        except Exception:
            pass

    need_padding = False
    if len(faces) > 0:
        face = max(faces, key=lambda f: (f["bbox"][2] - f["bbox"][0]) * (f["bbox"][3] - f["bbox"][1]))
        x1, y1, x2, y2 = face["bbox"]
        w_face, h_face = x2 - x1, y2 - y1
        occ_w = w_face / W
        occ_h = h_face / H
        if occ_w >= face_occupancy_thresh or occ_h >= face_occupancy_thresh or x1 < 0.05 * W or x2 > 0.95 * W or y1 < 0.05 * H or y2 > 0.95 * H:
            need_padding = True
    else:
        if min(H, W) < 300:
            need_padding = True

    if need_padding:
        pad_h = int(H * pad_ratio)
        pad_w = int(W * pad_ratio)
        cv_border = cv2.BORDER_REPLICATE if border_mode == "replicate" else cv2.BORDER_REFLECT_101
        padded = cv2.copyMakeBorder(image_rgb, pad_h, pad_h, pad_w, pad_w, cv_border)
        return padded, True
    return image_rgb, False


def apply_white_balance(
    image_rgb: np.ndarray,
    p: int = 6,                      # Minkowski p-norm (Shades of Gray, Finlayson & Trezzi 2004)
    max_shift_thresh: float = 35.0,  # Ngưỡng lệch kênh màu tối đa (30-40 đơn vị)
    gain_min: float = 0.75,          # Giới hạn giảm kênh tối đa 25%
    gain_max: float = 1.30,          # Giới hạn tăng kênh tối đa 30%
    return_info: bool = False
) -> np.ndarray:
    """
    Bước 2: Shades of Gray White Balance (Minkowski p-norm, p=6, Finlayson & Trezzi 2004)
    kết hợp giới hạn gain [0.75, 1.30] & Dynamic Alpha Blending.
    Trọng số dồn về vùng sáng nhất (specular highlights), triệt tiêu vệt xanh lá ở trán
    trên ảnh sepia đậm (003A35.JPG, 004A37.JPG) và bảo toàn sắc da ấm tự nhiên.
    """
    # 1. Chuyển sang float64 và chuẩn hóa về [0, 1] trước khi tính pixel^p tránh tràn số
    img_norm = image_rgb.astype(np.float64) / 255.0

    # 2. Tính Minkowski p-norm (p=6) cho từng kênh: avg_c = (1/N * sum(pixel^p))^(1/p)
    norm_r = np.power(np.mean(np.power(img_norm[:, :, 0], p)), 1.0 / p)
    norm_g = np.power(np.mean(np.power(img_norm[:, :, 1], p)), 1.0 / p)
    norm_b = np.power(np.mean(np.power(img_norm[:, :, 2], p)), 1.0 / p)

    avg_gray = (norm_r + norm_g + norm_b) / 3.0

    # 3. Tính raw gains
    raw_gain_r = float(avg_gray / (norm_r + 1e-8))
    raw_gain_g = float(avg_gray / (norm_g + 1e-8))
    raw_gain_b = float(avg_gray / (norm_b + 1e-8))

    # 4. Kẹp gains vào khoảng an toàn [gain_min, gain_max]
    clamped_gain_r = float(np.clip(raw_gain_r, gain_min, gain_max))
    clamped_gain_g = float(np.clip(raw_gain_g, gain_min, gain_max))
    clamped_gain_b = float(np.clip(raw_gain_b, gain_min, gain_max))

    # 5. Áp dụng gains đã kẹp lên ảnh gốc float32
    img_float = image_rgb.astype(np.float32)
    wb_temp = np.zeros_like(img_float)
    wb_temp[:, :, 0] = np.clip(img_float[:, :, 0] * clamped_gain_r, 0, 255)
    wb_temp[:, :, 1] = np.clip(img_float[:, :, 1] * clamped_gain_g, 0, 255)
    wb_temp[:, :, 2] = np.clip(img_float[:, :, 2] * clamped_gain_b, 0, 255)

    # 6. Đo độ lệch màu thực tế trước và sau WB
    avg_r = float(np.mean(img_float[:, :, 0]))
    avg_g = float(np.mean(img_float[:, :, 1]))
    avg_b = float(np.mean(img_float[:, :, 2]))

    shift_r = abs(float(np.mean(wb_temp[:, :, 0])) - avg_r)
    shift_g = abs(float(np.mean(wb_temp[:, :, 1])) - avg_g)
    shift_b = abs(float(np.mean(wb_temp[:, :, 2])) - avg_b)
    max_shift = max(shift_r, shift_g, shift_b)

    # 7. Dynamic Alpha Blending nếu max_shift > max_shift_thresh
    if max_shift > max_shift_thresh:
        alpha = max_shift_thresh / (max_shift + 1e-6)
    else:
        alpha = 1.0

    blended = img_float * (1.0 - alpha) + wb_temp * alpha
    out_rgb = np.clip(blended, 0, 255).astype(np.uint8)

    info = {
        "p_norm": (round(float(norm_r), 4), round(float(norm_g), 4), round(float(norm_b), 4)),
        "avg_orig": (round(avg_r, 1), round(avg_g, 1), round(avg_b, 1)),
        "raw_gains": (round(raw_gain_r, 3), round(raw_gain_g, 3), round(raw_gain_b, 3)),
        "clamped_gains": (round(clamped_gain_r, 3), round(clamped_gain_g, 3), round(clamped_gain_b, 3)),
        "max_shift": round(max_shift, 1),
        "alpha": round(alpha, 2),
        "avg_out": (round(float(out_rgb[:, :, 0].mean()), 1),
                    round(float(out_rgb[:, :, 1].mean()), 1),
                    round(float(out_rgb[:, :, 2].mean()), 1))
    }
    if return_info:
        return out_rgb, info
    return out_rgb


def run_codeformer(
    image_rgb: np.ndarray,
    fidelity_weight: float = 0.7,
    unique_tag: str = "temp",
    temp_dir: str = "/kaggle/working/temp_codeformer"
) -> np.ndarray:
    """
    Bước 3: Chạy CodeFormer inference trên 1 ảnh RGB với fidelity_weight cho trước (mặc định w=0.7).
    Gọi CLI inference_codeformer.py --face_upsample như đã validate trên tập mẫu.
    Trả về ảnh RGB sau phục hồi (np.ndarray uint8). Raise Exception nếu thất bại để kích hoạt fallback.
    """
    codeformer_dir = globals().get("CODEFORMER_DIR", "/kaggle/working/CodeFormer")
    codeformer_script = os.path.join(codeformer_dir, "inference_codeformer.py")
    if not os.path.exists(codeformer_script):
        for alt_d in ["/kaggle/working/CodeFormer", "./CodeFormer", "../CodeFormer"]:
            cand = os.path.join(alt_d, "inference_codeformer.py")
            if os.path.exists(cand):
                codeformer_script = cand
                break

    if not os.path.exists(codeformer_script):
        raise FileNotFoundError(f"Không tìm thấy script CodeFormer tại: {codeformer_script}")

    in_dir = os.path.join(temp_dir, f"in_{unique_tag}")
    out_dir = os.path.join(temp_dir, f"out_{unique_tag}")
    os.makedirs(in_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    in_file = os.path.join(in_dir, "input.png")
    cv2.imwrite(in_file, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

    cmd = [
        sys.executable, codeformer_script,
        "-w", str(fidelity_weight),
        "--input_path", in_file,
        "-o", out_dir,
        "--face_upsample"
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
    if proc.returncode != 0:
        raise RuntimeError(f"CodeFormer CLI lỗi (code {proc.returncode}): {proc.stderr[:200]}")

    res_path = os.path.join(out_dir, "final_results", "input.png")
    if not os.path.exists(res_path):
        fin_dir = os.path.join(out_dir, "final_results")
        if os.path.exists(fin_dir) and os.listdir(fin_dir):
            res_path = os.path.join(fin_dir, os.listdir(fin_dir)[0])
        else:
            raise FileNotFoundError(f"Không tìm thấy kết quả CodeFormer tại: {out_dir}")

    res_bgr = cv2.imread(res_path)
    if res_bgr is None:
        raise ValueError(f"Không đọc được ảnh kết quả CodeFormer: {res_path}")

    # Dọn dẹp thư mục tạm để tiết kiệm dung lượng đĩa
    try:
        shutil.rmtree(in_dir, ignore_errors=True)
        shutil.rmtree(out_dir, ignore_errors=True)
    except Exception:
        pass

    return cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)

# ==============================================================================
# 2. CĂN CHỈNH KHUÔN MẶT & HẬU XỬ LÝ BLENDING
# ==============================================================================

def align_to_ffhq(image_path: str, kps: np.ndarray, output_size: int = 512) -> Image.Image:
    """Align mot anh theo dung cong thuc goc NVIDIA (ffhq_dataset/face_alignment.py),
    dung 4 diem tu 5-point landmarks InsightFace (bo qua diem mui):
    mat trai, mat phai, khoe mieng trai, khoe mieng phai.
    Tich hop pad-and-blur (NVIDIA NVlabs) khi quad vuot bien de triet tieu vien den/hoa van hinh thoi."""
    eye_left = np.array(kps[0], dtype=np.float64)
    eye_right = np.array(kps[1], dtype=np.float64)
    mouth_left = np.array(kps[3], dtype=np.float64)
    mouth_right = np.array(kps[4], dtype=np.float64)

    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    qsize = max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    x *= qsize
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])

    # Đọc ảnh (hỗ trợ cả str filepath, PIL.Image hoặc numpy ndarray)
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    elif isinstance(image_path, Image.Image):
        img = image_path.convert("RGB")
    elif isinstance(image_path, np.ndarray):
        img = Image.fromarray(image_path).convert("RGB")
    else:
        raise ValueError(f"Unsupported image type: {type(image_path)}")
    img_w, img_h = img.size

    # ===== MỚI: Tính padding cần thiết nếu quad vượt biên (Thuật toán chuẩn FFHQ / NVlabs) =====
    border = max(int(round(qsize * 0.1)), 3)
    pad = (
        max(int(-np.floor(quad[:, 0].min())) + border, 0),
        max(int(-np.floor(quad[:, 1].min())) + border, 0),
        max(int(np.ceil(quad[:, 0].max())) - img_w + border, 0),
        max(int(np.ceil(quad[:, 1].max())) - img_h + border, 0)
    )

    if max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        pad_left, pad_top, pad_right, pad_bottom = pad

        # Vượt biên đáng kể -> mở rộng canvas bằng phản chiếu + làm mờ vùng nối
        img_np = np.pad(
            np.float32(np.array(img)),
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="reflect",
        )
        h, w, _ = img_np.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad_left, np.float32(w - 1 - x_grid) / pad_right),
            1.0 - np.minimum(np.float32(y_grid) / pad_top, np.float32(h - 1 - y_grid) / pad_bottom),
        )
        blur_sigma = qsize * 0.02
        img_np += (gaussian_filter(img_np, [blur_sigma, blur_sigma, 0]) - img_np) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img_np += (np.median(img_np, axis=(0, 1)) - img_np) * np.clip(mask, 0.0, 1.0)
        img = Image.fromarray(np.uint8(np.clip(np.rint(img_np), 0, 255)))
        # Dời quad theo đúng offset padding vừa thêm
        quad = quad + np.array([pad_left, pad_top])

    img = img.transform((output_size, output_size), Image.QUAD, (quad + 0.5).flatten(), Image.BILINEAR)
    return img


def align_image_for_pipeline(image_path: str, embedder, output_dir: str, unique_tag: str) -> str:
    """Ham tien ich: detect mat, align, LUU ra file voi TEN DUY NHAT, tra ve duong dan
    de dua vao inverter.invert(). Raise ValueError ro rang neu khong detect duoc mat."""
    faces = embedder.detect_faces(image_path)
    if len(faces) == 0:
        raise ValueError(f"Khong phat hien khuon mat de align: {image_path}")
    aligned = align_to_ffhq(image_path, faces[0]["kps"])
    os.makedirs(output_dir, exist_ok=True)
    aligned_path = os.path.join(output_dir, f"aligned_{unique_tag}.png")
    aligned.save(aligned_path)
    return aligned_path


def apply_mask_blending(original_image_path: str, generated_image: Image.Image, embedder) -> Image.Image:
    """
    Hậu xử lý Mask-based Blending: Sử dụng landmark của InsightFace để tạo mặt nạ mềm (soft mask)
    cho vùng khuôn mặt, sau đó pha trộn ảnh đã edit vào ảnh gốc để giữ trọn phông nền, tóc và áo quần.
    """
    orig_img = cv2.imread(original_image_path)
    if orig_img is None:
        return generated_image
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    gen_np = np.array(generated_image.resize((orig_img.shape[1], orig_img.shape[0]), Image.BICUBIC))
    
    faces = embedder.detect_faces(original_image_path)
    if len(faces) == 0:
        return generated_image
        
    face = faces[0]
    kps = face["kps"]
    
    mask = np.zeros(orig_img.shape[:2], dtype=np.float32)
    bbox = face["bbox"].astype(int)
    x1, y1, x2, y2 = bbox
    w, h = x2 - x1, y2 - y1
    
    x1 = max(0, int(x1 - 0.1 * w))
    x2 = min(orig_img.shape[1], int(x2 + 0.1 * w))
    y1 = max(0, int(y1 - 0.2 * h))
    y2 = min(orig_img.shape[0], int(y2 + 0.1 * h))
    
    center = ((x1 + x2) // 2, (y1 + y2) // 2)
    axes = (int((x2 - x1) * 0.55), int((y2 - y1) * 0.6))
    cv2.ellipse(mask, center, axes, 0, 0, 360, 1.0, -1)
    
    blur_kernel = int(min(w, h) * 0.25)
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    mask = cv2.GaussianBlur(mask, (blur_kernel, blur_kernel), 0)
    mask_3ch = np.stack([mask, mask, mask], axis=-1)
    
    blended = orig_img.astype(np.float32) * (1.0 - mask_3ch) + gen_np.astype(np.float32) * mask_3ch
    blended = np.clip(blended, 0, 255).astype(np.uint8)
    return Image.fromarray(blended)


## Khởi tạo Pipeline Models (Inverter, Editor, Embedder — Khởi tạo 1 lần duy nhất)

Nạp sẵn UNet đã fine-tune, SD v1.5 components, và InsightFace vào bộ nhớ. Toàn bộ các cell Demo và Đánh giá FG-NET bên dưới sẽ dùng chung instance này, không khởi tạo lại để tiết kiệm thời gian và bộ nhớ.


In [ ]:
ckpt_dir = config["paths"]["specialized_unet_ckpt"]
print("Khởi tạo Inverter, Editor và Embedder dùng chung cho toàn bộ pipeline...")
inverter = NullTextInverter(
    pretrained_model_name_or_path=config["base_model"]["pretrained_model_name_or_path"],
    unet_checkpoint_dir=ckpt_dir,
    num_inference_steps=config["inversion"]["num_inference_steps"],
    guidance_scale=config["inversion"]["guidance_scale"],
    num_inner_steps=config["inversion"]["num_inner_steps"],
    early_stop_epsilon=float(config["inversion"]["early_stop_epsilon"]),
    image_size=config["inversion"]["image_size"],
)
editor = Editor(
    pretrained_model_name_or_path=config["base_model"]["pretrained_model_name_or_path"],
    unet_checkpoint_dir=ckpt_dir,
    num_inference_steps=config["inversion"]["num_inference_steps"],
    guidance_scale=config["editing"]["guidance_scale"],
    attention_control_ratio=config["editing"]["attention_control_ratio"],
    image_size=config["editing"]["image_size"],
)
embedder = FaceEmbedder(
    model_name=config["embedding"]["model_name"],
    ctx_id=config["embedding"]["ctx_id"],
    det_size=tuple(config["embedding"]["det_size"]),
)
print("✅ Đã sẵn sàng: inverter, editor, embedder.")


## Thử nghiệm ảnh đơn (Single Image Demo)


In [ ]:
import pandas as pd
from PIL import Image
# Chọn ảnh test
TEST_IMAGE_NAME = "01366.png"
TEST_IMAGE_PATH = os.path.join(FFHQ_DIR, TEST_IMAGE_NAME)
# Tra đúng label thật của ảnh này từ CSV - KHÔNG hardcode đoán tuổi/giới tính nữa
df = pd.read_csv(LABELS_CSV)
image_number = int(TEST_IMAGE_NAME.replace(".png", ""))
row = df[df["image_number"] == image_number].iloc[0]
INITIAL_AGE = age_group_to_age(row["age_group"])
GENDER_WORD = gender_to_word(row["gender"], INITIAL_AGE)
TARGET_AGES = [20, 30, 40, 50, 60, 70, 80]   # mục tiêu tự chọn tay, không liên quan label gốc
print(f"Ảnh: {TEST_IMAGE_NAME} | age_group thật: {row['age_group']} | gender thật: {row['gender']}")
print(f"=> INITIAL_AGE={INITIAL_AGE}, GENDER_WORD='{GENDER_WORD}'")
# Căn chỉnh ảnh test trước khi đưa vào Invert
test_aligned_dir = os.path.join(OUTPUT_DIR, "test_aligned")
ALIGNED_TEST_IMAGE_PATH = align_image_for_pipeline(
    TEST_IMAGE_PATH, embedder, test_aligned_dir, "demo_01366"
)
print(f"✅ Đã căn chỉnh ảnh test theo chuẩn FFHQ 512x512: {ALIGNED_TEST_IMAGE_PATH}")
print("\nẢnh input test (đã align 512x512):")
display(Image.open(ALIGNED_TEST_IMAGE_PATH))


### Chạy Null-text Inversion (Module 2)


In [ ]:
print("Chay Module 2 (Null-text Inversion)...")
# Chạy Inversion trên ảnh đã căn chỉnh 512x512
z_T, null_embeddings, (self_maps, cross_maps) = inverter.invert(
    ALIGNED_TEST_IMAGE_PATH, INITIAL_AGE, GENDER_WORD
)
# Kiểm tra NaN cho cả Self và Cross Attention maps
_nan_found_attn = False
for maps_dict, tag in [(self_maps, "self_maps"), (cross_maps, "cross_maps")]:
    for _t, _layers in maps_dict.items():
        for _name, _tensor in _layers.items():
            if torch.isnan(_tensor).any():
                print(f"  ❌ {tag}[t={_t}][{_name}] CO NaN!")
                _nan_found_attn = True
if not _nan_found_attn:
    print("  ✅ Khong co NaN trong ca self_maps va cross_maps")
print("  ✅ z_T binh thuong" if not torch.isnan(z_T).any() else "  ❌ z_T CO NaN!")
# Giữ inverter trong bộ nhớ cho các tác vụ tiếp theo
torch.cuda.empty_cache()
print("z_T shape:", z_T.shape)
print("so luong null_t:", len(null_embeddings))
print(f"so timestep: self_maps={len(self_maps)}, cross_maps={len(cross_maps)}")
attention_maps = (self_maps, cross_maps)


In [ ]:
# ==============================================================================
# BƯỚC BẮT BUỘC: KIỂM TRA TÁI TẠO (RECONSTRUCTION SANITY-CHECK)
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
print("Đang chạy kiểm tra tái tạo ảnh gốc từ z_T và null_embeddings...")
# 0. Giải mã thử z0 trực tiếp để xác nhận VAE không làm mất phông nền và áo sọc
if editor.vae is None:
    editor._load_models()
direct_z0_img = editor._decode_latent_to_image(inverter._load_image_latent(ALIGNED_TEST_IMAGE_PATH))

# 1. Tái tạo ảnh bằng prompt gốc P_alpha từ z_T
reconstructed_img = editor.reconstruct(
    z_T=z_T,
    null_embeddings=null_embeddings,
    initial_age=INITIAL_AGE,
    gender_word=GENDER_WORD,
    guidance_scale=config["inversion"]["guidance_scale"],  # Khử nhiễu thuần túy ODE (1.0) giữ 100% phông nền & áo sọc
)

# 2. Lưu ảnh tạm thời để đưa qua FaceEmbedder đo ID Score
temp_rec_path = os.path.join(OUTPUT_DIR, "sanity_check_reconstruction.png")
reconstructed_img.save(temp_rec_path)

# 3. Đo độ tương đồng nhận diện giữa ảnh gốc (đã align) và ảnh tái tạo
orig_emb = embedder.embed(ALIGNED_TEST_IMAGE_PATH)
rec_emb = embedder.embed(temp_rec_path)
rec_id_score = float(np.dot(orig_emb, rec_emb))

# 4. Hiển thị 3 ảnh đối chứng: Ảnh gốc -> Giải mã VAE trực tiếp -> Tái tạo từ Inversion
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(ALIGNED_TEST_IMAGE_PATH))
axes[0].set_title("1. Ảnh gốc đã Align", fontsize=11)
axes[0].axis("off")

axes[1].imshow(direct_z0_img)
axes[1].set_title("2. Giải mã VAE trực tiếp từ z0\n(Chuẩn đối chứng 100% nét thật)", fontsize=11)
axes[1].axis("off")

axes[2].imshow(reconstructed_img)
axes[2].set_title(f"3. Tái tạo từ Inversion\nID Score: {rec_id_score:.4f}", fontsize=11)
axes[2].axis("off")
plt.tight_layout()
plt.show()
# 5. Đánh giá chất lượng và cảnh báo
print(f"-> Cosine Similarity (ID Score): {rec_id_score:.4f}")
if rec_id_score >= 0.60:
    print("✅ XÁC THỰC THÀNH CÔNG: Inversion bảo toàn xuất sắc đặc trưng khuôn mặt.")
elif rec_id_score >= 0.40:
    print("✅ ĐẠT YÊU CẦU: Inversion đạt chuẩn tái tạo hình học của Diffusion (0.40 - 0.60). Đủ điều kiện chạy tiếp các Module sau.")
else:
    print("⚠️ CẢNH BÁO: Điểm tái tạo dưới 0.40, cần kiểm tra lại độ căn chỉnh góc mặt.")
    print("   KHUYẾN NGHỊ: Không chạy tiếp Module 3. Cần kiểm tra lại:")
    print("   - Đảm bảo ảnh test đã được align_to_ffhq đúng chuẩn 512x512.")
    print("   - Tăng num_inner_steps trong config['inversion'] từ 10 lên 15 hoặc 20.")
    print("   - Đảm bảo optimizer uncond_embeddings được giữ ở FP32 khi tính gradient.")


### Chạy Editing (Module 3)


In [ ]:
print("Chay Module 3 (Editing)...")
# Sử dụng ảnh đã align làm tham chiếu gốc cho mask blending để bảo toàn phông nền, tóc và trang phục
ref_align_path = ALIGNED_TEST_IMAGE_PATH if "ALIGNED_TEST_IMAGE_PATH" in locals() else TEST_IMAGE_PATH
edited_images = editor.edit(
    z_T=z_T,
    null_embeddings=null_embeddings,
    attention_maps=attention_maps,
    target_ages=TARGET_AGES,
    gender_word=GENDER_WORD,
    output_dir=config["paths"]["output_dir"],
    original_image_path=ref_align_path,
    embedder=embedder,
    use_local_blend=True,       # Bật lại LocalBlend để bảo toàn vùng ngoài chủ thể tại mỗi bước denoising
    initial_age=INITIAL_AGE,    # Truyền độ tuổi gốc phục vụ tái tạo latent reconstruction cho LocalBlend
)

# Giữ editor trong bộ nhớ cho các tác vụ tiếp theo
torch.cuda.empty_cache()

edited_images


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
print("Đang tính toán Cosine Similarity (ID Score) cho các mốc tuổi từ 20 đến 80...")
ref_image_path = ALIGNED_TEST_IMAGE_PATH if "ALIGNED_TEST_IMAGE_PATH" in locals() else TEST_IMAGE_PATH
orig_emb = embedder.embed(ref_image_path)
fig, axes = plt.subplots(1, len(TARGET_AGES), figsize=(4 * len(TARGET_AGES), 4))
if len(TARGET_AGES) == 1:
    axes = [axes]
for ax, age in zip(axes, TARGET_AGES):
    path = edited_images[age]
    img = Image.open(path)
    
    # Tính ID Score cho ảnh ở mốc tuổi này
    gen_emb = embedder.embed(path)
    id_score = float(np.dot(orig_emb, gen_emb))
    
    ax.imshow(img)
    ax.set_title(f"Age: {age}\nID Score: {id_score:.4f}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("\n--- BẢNG ID SCORE CHI TIẾT THEO MỐC TUỔI ---")
for age in TARGET_AGES:
    path = edited_images[age]
    gen_emb = embedder.embed(path)
    score = float(np.dot(orig_emb, gen_emb))
    print(f"  - Target Age {age}: ID Score = {score:.4f}")


## Chuẩn bị dữ liệu FG-NET cho đánh giá

Parse thư mục ảnh FG-NET, tạo các cặp tuổi nguồn - đích (source_age < target_age) để đánh giá quá trình già hóa.


In [ ]:
import re
from collections import defaultdict

assert fgnet_images_dir is not None and os.path.isdir(fgnet_images_dir), (
    "❌ Không tìm thấy thư mục ảnh FG-NET. Kiểm tra lại cấu hình fgnet_images_dir ở Cell 1."
)
print(f"Sử dụng thư mục ảnh FG-NET: {fgnet_images_dir}")

# Parse tên file dạng "001A05.JPG" -> người 001, tuổi 05
pattern = re.compile(r"(\d{3})A(\d{2})", re.IGNORECASE)
persons = defaultdict(list)
skipped = 0
for fname in sorted(os.listdir(fgnet_images_dir)):
    m = pattern.match(fname)
    if m:
        person_id, age = m.group(1), int(m.group(2))
        persons[person_id].append((os.path.join(fgnet_images_dir, fname), age))
    else:
        skipped += 1

print(f"\nTổng số người: {len(persons)}")
print(f"Tổng số ảnh parse được: {sum(len(v) for v in persons.values())}")
print(f"Số file không khớp pattern (bỏ qua): {skipped}")


In [ ]:
import random

# SUA: 3 -> 2 cap/nguoi - bu lai thoi gian da TANG GAP DOI do num_inference_steps
# 50->100 (tong thoi gian uoc tinh: ~2/3 x 2 = ~1.33 lan so voi ban goc 50 buoc/3 cap,
# thay vi ~2 lan neu giu nguyen 3 cap).
MAX_PAIRS_PER_PERSON = 2   # gioi han so cap/nguoi de kiem soat tong thoi gian chay
random.seed(42)

# SUA: CHI GIU CHIEU GIA HOA (source_age < target_age) - khop dung use-case that
# cua de tai (tim nguoi that lac: co anh CU luc con nho/tre, can suy ra HIEN TAI
# gia hon). Loc NGAY TU NGUON (truoc khi lay mau), khong loc sau, de moi nguoi
# van duoc lay du toi da MAX_PAIRS_PER_PERSON cap nhu cu.
all_pairs = []
for person_id, images in persons.items():
    if len(images) < 2:
        continue
    possible_pairs = [(a, b) for a in images for b in images if a != b and a[1] < b[1]]
    if len(possible_pairs) == 0:
        continue
    sampled = random.sample(possible_pairs, min(MAX_PAIRS_PER_PERSON, len(possible_pairs)))
    for (src_path, src_age), (tgt_path, tgt_age) in sampled:
        all_pairs.append({
            "person_id": person_id,
            "source_img": src_path, "source_age": src_age,
            "target_img": tgt_path, "target_age": tgt_age,
        })

print(f"Tong so cap se chay: {len(all_pairs)}")
print(f"(trung binh {len(all_pairs)/len(persons):.1f} cap/nguoi, tren {len(persons)} nguoi)")

# SUA: cap nhat lai uoc luong - ~130s/cap (gap doi 65s cu) do num_inference_steps
# da tang 50->100 (ca Inversion lan Editing deu chay nhieu buoc hon).
est_seconds = len(all_pairs) * 130
print(f"Uoc luong thoi gian chay: ~{est_seconds/60:.0f} phut (~{est_seconds/3600:.1f} tieng)")
print("Luu y: Kaggle co the ngat phien truoc khi chay xong - khong sao, cell duoi ho tro resume.")

# Lưu tổng số cặp ra file — để cell báo cáo cuối tính đúng tỷ lệ lỗi dù chạy ở
# phiên Kaggle khác (biến all_pairs có thể không còn trong bộ nhớ sau khi resume).
with open(os.path.join(OUTPUT_DIR, "fgnet_total_pairs.txt"), "w") as f:
    f.write(str(len(all_pairs)))

# ===== Build gallery FG-NET cho Rank-1 Accuracy =====
# Dùng TẤT CẢ ảnh thật sẵn có của 82 người (không chỉ ảnh target đã dùng tính
# id_score) làm gallery — mô phỏng đúng tình huống thật: 1 người có thể có nhiều
# ảnh trong database, tìm kiếm là so ảnh sinh ra với TOÀN BỘ database đó.
print("\nĐang build gallery FG-NET (toàn bộ ảnh thật của 82 người)...")
gallery_embeddings_fgnet = []
gallery_labels_fgnet = []
gallery_paths_fgnet = []  # v2: để loại ảnh nguồn khỏi gallery khi tính Rank-k
for person_id, images in persons.items():
    for img_path, _age in images:
        try:
            emb = embedder.embed(img_path)
            gallery_embeddings_fgnet.append(emb)
            gallery_labels_fgnet.append(person_id)
            gallery_paths_fgnet.append(img_path)
        except ValueError:
            continue
gallery_matrix_fgnet = np.stack(gallery_embeddings_fgnet)
print(f"✅ Gallery FG-NET: {len(gallery_labels_fgnet)} ảnh, {len(set(gallery_labels_fgnet))} người.")

# v2: lưu embedding gallery để phân tích lại trên máy (không cần GPU / sinh lại ảnh).
FGNET_EMB_DIR = os.path.join(OUTPUT_DIR, "fgnet_embeddings")
os.makedirs(FGNET_EMB_DIR, exist_ok=True)
np.savez_compressed(
    os.path.join(FGNET_EMB_DIR, "gallery.npz"),
    emb=gallery_matrix_fgnet.astype(np.float32),
    labels=np.array(gallery_labels_fgnet),
    paths=np.array([os.path.basename(p) for p in gallery_paths_fgnet]),
)
print(f"Đã lưu embedding gallery: {os.path.join(FGNET_EMB_DIR, 'gallery.npz')}")


## Đánh giá chính thức trên FG-NET — ID Score + Age MAE

Chạy trên **toàn bộ người** trong FG-NET (không chỉ 1 người), lấy mẫu 1 số cặp/người để kiểm soát thời gian. Kết quả ghi liên tục ra CSV trong OUTPUT_DIR — nếu bị ngắt phiên giữa chừng, chạy lại cell là **tự động resume** từ chỗ dang dở, không chạy lại từ đầu. ⚠️ Trên Kaggle, OUTPUT_DIR không bền vững giữa các phiên — tải CSV về máy định kỳ để không mất tiến độ.

## Age Estimator thật cho IA (MiVOLO) — thay thế tra cứu CSV khi dùng ảnh KHÔNG có nhãn sẵn

Phần "Nhập tay" ở đầu notebook tra `INITIAL_AGE` từ `sampled_labels.csv` — **chỉ hoạt động với 140 ảnh đã có sẵn nhãn**. Phần này thêm 1 cách khác: dùng model **MiVOLO** để tự đoán tuổi cho **bất kỳ ảnh nào** (VD ảnh thật của người mất tích, không có CSV đi kèm).

⚠️ **Lưu ý quan trọng**: API bên dưới dựa theo tài liệu chính thức của MiVOLO tại thời điểm viết, nhưng **chưa được tự chạy để xác nhận 100%** — tên tham số/đường dẫn checkpoint có thể đã đổi theo phiên bản. Nếu gặp lỗi, đối chiếu lại với `github.com/WildChlamydia/MiVOLO`.

In [ ]:
# ===== Tải / Kiểm tra checkpoint MiVOLO =====
from huggingface_hub import hf_hub_download

MIVOLO_DETECTOR_CKPT = None
MIVOLO_AGE_CKPT = None

# 1. Kiểm tra cache local hoặc dataset có sẵn trước
if os.path.isdir(MIVOLO_LOCAL_DIR):
    local_det = os.path.join(MIVOLO_LOCAL_DIR, MIVOLO_DETECTOR_FILE)
    if os.path.isfile(local_det):
        MIVOLO_DETECTOR_CKPT = local_det
    for _, fname in MIVOLO_AGE_CKPT_SOURCES:
        local_age = os.path.join(MIVOLO_LOCAL_DIR, os.path.basename(fname))
        if os.path.isfile(local_age):
            MIVOLO_AGE_CKPT = local_age
            break

# 2. Tải từ HuggingFace nếu chưa có
if MIVOLO_DETECTOR_CKPT is None:
    print("Đang tải checkpoint detector (người/mặt)...")
    MIVOLO_DETECTOR_CKPT = hf_hub_download(MIVOLO_DETECTOR_REPO, MIVOLO_DETECTOR_FILE)
print(f"  -> Detector: {MIVOLO_DETECTOR_CKPT}")

if MIVOLO_AGE_CKPT is None:
    print("Đang tải checkpoint age/gender model (nguồn mirror công khai)...")
    for repo_id, filename in MIVOLO_AGE_CKPT_SOURCES:
        try:
            MIVOLO_AGE_CKPT = hf_hub_download(repo_id, filename)
            print(f"  -> Tải thành công từ '{repo_id}': {MIVOLO_AGE_CKPT}")
            break
        except Exception as e:
            print(f"  Không tải được từ '{repo_id}': {type(e).__name__}: {e}")

if MIVOLO_AGE_CKPT is None:
    raise RuntimeError(
        "Không tải được checkpoint age/gender từ bất kỳ nguồn nào trong MIVOLO_AGE_CKPT_SOURCES."
    )

print("\n✅ Hoàn tất nạp checkpoint MiVOLO.")


In [ ]:
# ===== Khởi tạo MiVOLO predictor + hàm wrapper =====
import argparse
import cv2
import numpy as np

mivolo_predictor = None

def _load_mivolo():
    """Load MiVOLO predictor - chỉ load 1 lần (lazy), tái sử dụng cho các lần gọi sau."""
    global mivolo_predictor
    if mivolo_predictor is not None:
        return

    from mivolo.predictor import Predictor

    # PyTorch 2.6+ mở khóa weights_only=False cho checkpoint YOLO
    import torch
    if not getattr(torch.load, "_da_vo_hieu_weights_only", False):
        _torch_load_goc = torch.load
        def _torch_load_khong_weights_only(*args, **kwargs):
            kwargs.setdefault("weights_only", False)
            return _torch_load_goc(*args, **kwargs)
        _torch_load_khong_weights_only._da_vo_hieu_weights_only = True
        torch.load = _torch_load_khong_weights_only

    mivolo_args = argparse.Namespace(
        detector_weights=MIVOLO_DETECTOR_CKPT,
        checkpoint=MIVOLO_AGE_CKPT,
        device="cuda" if torch.cuda.is_available() else "cpu",
        with_persons=True,
        disable_faces=False,
        draw=False,
    )
    mivolo_predictor = Predictor(mivolo_args)
    print("✅ Đã khởi tạo MiVOLO predictor.")

def mivolo_estimate_age_and_gender(image_path: str):
    """
    Ước lượng cả tuổi và giới tính của người trong ảnh bằng MiVOLO.
    Trả về (estimated_age, estimated_gender_str).
    """
    _load_mivolo()
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")

    try:
        detected_objects, _ = mivolo_predictor.recognize(img)
    except Exception as e:
        raise RuntimeError(f"Lỗi khi chạy MiVOLO trên ảnh {image_path}: {e}")

    if not hasattr(detected_objects, "ages") or len(detected_objects.ages) == 0:
        raise ValueError(f"MiVOLO không phát hiện được khuôn mặt trong ảnh: {image_path}")

    est_age = round(detected_objects.ages[0])
    
    est_gender = "unknown"
    if hasattr(detected_objects, "genders") and len(detected_objects.genders) > 0:
        g_val = detected_objects.genders[0]
        if isinstance(g_val, (int, np.integer)):
            est_gender = "female" if g_val == 0 else "male"
        else:
            est_gender = str(g_val).lower()

    return est_age, est_gender

def mivolo_estimate_age(image_path: str) -> int:
    """Ước lượng tuổi của người trong ảnh bằng MiVOLO."""
    age, _ = mivolo_estimate_age_and_gender(image_path)
    return age


### Giao thức đánh giá v2 (sửa rò rỉ & lệch pipeline)

1. **Rank-k loại ảnh nguồn khỏi gallery** — ảnh sinh dựng từ ảnh nguồn, nên nếu ảnh nguồn còn trong gallery thì Rank-1 bị thổi phồng. Cột `rank1_gen_leaky` giữ cách tính cũ để đối chiếu.
2. **Baseline cùng pipeline**: `baseline_cos` = cosine(ảnh nguồn gốc, ảnh đích) tính bằng đúng embedder dùng cho ảnh sinh.
3. **Kết hợp**: `fused_max_cos`, `fused_mean_cos`, `rank1_fused_*` — dùng ảnh sinh *bổ sung* cho ảnh gốc thay vì thay thế.
4. **Giữ ảnh sinh theo cặp** (`fgnet_generated_pairs/`) — editor lưu `age_{tuổi}.png` nên bản cũ bị ghi đè; đặt `REUSE_GENERATED_DIR` để dùng lại ảnh đã sinh, bỏ qua bước diffusion.
5. **Lưu embedding** (`fgnet_embeddings/`: gallery.npz + gen/src/tgt .npy mỗi cặp) để phân tích lại trên máy, không cần GPU.

Kết quả ghi vào `fgnet_eval_results_v2.csv` (không lẫn với CSV cũ).


In [ ]:
import csv
import shutil
import time
from datetime import timedelta
import matplotlib.pyplot as plt
from PIL import Image

# v2: file MỚI — không ghi nối vào CSV cũ (khác cột, và CSV cũ có Rank-1 rò rỉ).
RESULTS_CSV = os.path.join(OUTPUT_DIR, "fgnet_eval_results_v2.csv")
FIELDNAMES = [
    "person_id", "source_img", "source_age", "target_img", "target_age",
    "id_score", "estimated_age", "age_mae", "estimated_gender", "gender_match",
    "preprocessing_fallback", "runtime_sec", "rank1_correct", "top1_identity",
    # v2 — so sánh ghép cặp trên CÙNG pipeline embedding:
    "baseline_cos",       # cosine(ảnh nguồn gốc, ảnh đích)  — truy vấn KHÔNG dùng ảnh sinh
    "fused_max_cos",      # max(id_score, baseline_cos)
    "fused_mean_cos",     # cosine(mean(gen, src) chuẩn hoá, ảnh đích)
    "rank1_baseline", "rank5_baseline",      # gallery ĐÃ LOẠI ảnh nguồn
    "rank5_correct",                          # ảnh sinh, gallery đã loại ảnh nguồn
    "rank1_fused_max", "rank1_fused_mean",
    "rank1_gen_leaky",                        # cách tính CŨ (gallery chứa ảnh nguồn) — đối chiếu
    "generated_path",
]

# Thư mục cho ảnh sinh ra trong vòng lặp đánh giá FG-NET
FGNET_TEMP_DIR = os.path.join(OUTPUT_DIR, "fgnet_eval_temp_images")
os.makedirs(FGNET_TEMP_DIR, exist_ok=True)

# Thư mục cho ảnh SOURCE ĐÃ TIỀN XỬ LÝ (Pad + WB + CodeFormer)
FGNET_PREPROCESSED_DIR = os.path.join(OUTPUT_DIR, "fgnet_preprocessed_sources")
os.makedirs(FGNET_PREPROCESSED_DIR, exist_ok=True)

# Thư mục cho ảnh ĐÃ ALIGN (chuẩn 512x512 đưa vào Inverter)
FGNET_ALIGNED_DIR = os.path.join(OUTPUT_DIR, "fgnet_aligned_inputs")
os.makedirs(FGNET_ALIGNED_DIR, exist_ok=True)

# v2: editor luôn lưu "age_{target_age}.png" → cặp sau GHI ĐÈ ảnh cặp trước.
# Chép ảnh sinh ra tên riêng theo cặp để giữ lại, và dùng lại nếu đã có (resume/tái chạy).
FGNET_GEN_PAIRS_DIR = os.path.join(OUTPUT_DIR, "fgnet_generated_pairs")
os.makedirs(FGNET_GEN_PAIRS_DIR, exist_ok=True)
# Tùy chọn: thư mục ảnh sinh từ phiên trước (upload thành Kaggle dataset) để BỎ QUA
# bước sinh ảnh — chỉ tính lại metrics. Để None nếu không có.
REUSE_GENERATED_DIR = None

FGNET_EMB_DIR = os.path.join(OUTPUT_DIR, "fgnet_embeddings")
os.makedirs(FGNET_EMB_DIR, exist_ok=True)

gallery_paths_arr = np.array(gallery_paths_fgnet)
gallery_labels_arr = np.array(gallery_labels_fgnet)


def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def _rank_hits(sims, source_path, person_id, k=5):
    """Rank-1 / Rank-k trên gallery ĐÃ LOẠI chính ảnh nguồn (tránh rò rỉ)."""
    sims = sims.astype(np.float64).copy()
    sims[gallery_paths_arr == source_path] = -np.inf
    order = np.argsort(-sims)
    top = gallery_labels_arr[order[:k]]
    return bool(top[0] == person_id), bool(person_id in top), str(top[0])

# Cấu hình hiển thị trực quan inline (4 ảnh: source gốc -> source tiền xử lý -> generated -> target thật)
SHOW_INLINE_IMAGES = True
INLINE_DISPLAY_FREQ = 1  # 1 = hiển thị từng cặp, hoặc đặt N để hiển thị mỗi N cặp

# ----- Đọc lại các cặp ĐÃ chạy từ phiên trước (nếu có) để resume -----
done_pairs = set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            done_pairs.add((row["source_img"], row["target_img"]))
    print(f"Tìm thấy {len(done_pairs)} cặp đã chạy từ phiên trước -> sẽ bỏ qua, chỉ chạy phần còn lại.\n")

remaining_pairs = [p for p in all_pairs if (p["source_img"], p["target_img"]) not in done_pairs]
print(f"Số cặp còn lại cần chạy: {len(remaining_pairs)} / {len(all_pairs)}")
print("=" * 75)

write_header = not os.path.exists(RESULTS_CSV)

running_id_scores = []
running_age_maes = []
n_success = 0
n_failed = 0
n_fallback = 0
t_start = time.time()

with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if write_header:
        writer.writeheader()

    for idx, pair in enumerate(remaining_pairs, start=1):
        key = (pair["source_img"], pair["target_img"])
        t_pair_start = time.time()

        try:
            gender_word = "person"
            unique_tag = f"p{pair['person_id']}_{pair['source_age']}"
            
            # ====================================================================
            # 1. ĐỌC ẢNH SOURCE ĐÃ TIỀN XỬ LÝ (TỪ DATASET BATCH ĐÃ CHẠY SẴN)
            # ====================================================================
            source_filename = os.path.basename(pair["source_img"])
            preprocessed_source_path = os.path.join(PREPROCESSED_FGNET_DIR, source_filename)
            preprocessing_fallback = False

            if not os.path.exists(preprocessed_source_path):
                print(f"⚠️ Không tìm thấy ảnh đã tiền xử lý cho {pair['source_img']}, dùng ảnh gốc")
                preprocessed_source_path = pair["source_img"]
                preprocessing_fallback = True
                n_fallback += 1

            try:
                prep_bgr = cv2.imread(preprocessed_source_path)
                if prep_bgr is not None:
                    preprocessed_img_for_display = cv2.cvtColor(prep_bgr, cv2.COLOR_BGR2RGB)
                else:
                    orig_bgr = cv2.imread(pair["source_img"])
                    preprocessed_img_for_display = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB) if orig_bgr is not None else None
            except Exception:
                preprocessed_img_for_display = None

            pair_name = f"{pair['person_id']}_{_stem(pair['source_img'])}_to_{_stem(pair['target_img'])}.png"
            generated = os.path.join(FGNET_GEN_PAIRS_DIR, pair_name)
            reuse = REUSE_GENERATED_DIR and os.path.isfile(os.path.join(REUSE_GENERATED_DIR, pair_name))
            if reuse:
                shutil.copyfile(os.path.join(REUSE_GENERATED_DIR, pair_name), generated)
            elif not os.path.isfile(generated):
                # ====================================================================
                # 2. CĂN CHỈNH KHUÔN MẶT (ALIGN TO FFHQ 512x512)
                # ====================================================================
                aligned_source_path = align_image_for_pipeline(
                    preprocessed_source_path, embedder, FGNET_ALIGNED_DIR, unique_tag
                )

                # ====================================================================
                # 3. MODULE 2: NULL-TEXT INVERSION (100 bước DDIM)
                # ====================================================================
                z_T, null_t, attn_maps = inverter.invert(aligned_source_path, pair["source_age"], gender_word)

                # ====================================================================
                # 4. MODULE 3: EDITING với LocalBlend & Mask-based Blending
                # ====================================================================
                gen_tmp = editor.edit(
                    z_T, null_t, attn_maps,
                    target_ages=[pair["target_age"]],
                    gender_word=gender_word,
                    output_dir=FGNET_TEMP_DIR,
                    original_image_path=aligned_source_path,
                    embedder=embedder,
                    use_local_blend=True,           # Bật LocalBlend bảo toàn vùng nền, tóc, áo
                    initial_age=pair["source_age"],
                )[pair["target_age"]]
                shutil.copyfile(gen_tmp, generated)

            # ====================================================================
            # 5. ĐÁNH GIÁ ĐỊNH LƯỢNG — cùng 1 embedder cho ảnh sinh, ảnh nguồn, ảnh đích
            # ====================================================================
            gen_emb = embedder.embed(generated)
            tgt_emb = embedder.embed(pair["target_img"]) # SO SÁNH VỚI ẢNH ĐÍCH NGUYÊN BẢN
            src_emb = embedder.embed(pair["source_img"]) # ẢNH NGUỒN GỐC (baseline)
            id_score = float((gen_emb * tgt_emb).sum())
            baseline_cos = float((src_emb * tgt_emb).sum())
            fused_vec = gen_emb + src_emb
            fused_vec = fused_vec / np.linalg.norm(fused_vec)
            fused_mean_cos = float((fused_vec * tgt_emb).sum())
            fused_max_cos = max(id_score, baseline_cos)

            stem = pair_name[:-4]
            np.save(os.path.join(FGNET_EMB_DIR, f"{stem}__gen.npy"), gen_emb.astype(np.float32))
            np.save(os.path.join(FGNET_EMB_DIR, f"{stem}__src.npy"), src_emb.astype(np.float32))
            np.save(os.path.join(FGNET_EMB_DIR, f"{stem}__tgt.npy"), tgt_emb.astype(np.float32))

            estimated_age, estimated_gender = mivolo_estimate_age_and_gender(generated)
            age_mae = abs(estimated_age - pair["target_age"])
            try:
                _, src_gender = mivolo_estimate_age_and_gender(pair["source_img"])
                gender_match = (estimated_gender == src_gender)
            except Exception:
                gender_match = True

            # ====================================================================
            # 5b. RANK-k trong gallery FG-NET — ĐÃ LOẠI ảnh nguồn (v2)
            # Ảnh sinh dựng từ ảnh nguồn → nếu ảnh nguồn còn trong gallery thì
            # rank-1 bị thổi phồng (bản cũ). rank1_gen_leaky giữ cách cũ để đối chiếu.
            # ====================================================================
            sims_gen = gallery_matrix_fgnet @ gen_emb
            sims_src = gallery_matrix_fgnet @ src_emb
            rank1_correct_fgnet, rank5_gen, top1_identity_fgnet = _rank_hits(
                sims_gen, pair["source_img"], pair["person_id"])
            rank1_base, rank5_base, _ = _rank_hits(sims_src, pair["source_img"], pair["person_id"])
            rank1_fmax, _, _ = _rank_hits(np.maximum(sims_gen, sims_src),
                                          pair["source_img"], pair["person_id"])
            rank1_fmean, _, _ = _rank_hits(gallery_matrix_fgnet @ fused_vec,
                                           pair["source_img"], pair["person_id"])
            rank1_gen_leaky = gallery_labels_fgnet[int(np.argmax(sims_gen))] == pair["person_id"]

            runtime_sec = time.time() - t_pair_start

            writer.writerow({
                **pair,
                "id_score": id_score,
                "estimated_age": estimated_age,
                "age_mae": age_mae,
                "estimated_gender": estimated_gender,
                "gender_match": gender_match,
                "preprocessing_fallback": preprocessing_fallback,
                "runtime_sec": runtime_sec,
                "rank1_correct": rank1_correct_fgnet,
                "top1_identity": top1_identity_fgnet,
                "baseline_cos": baseline_cos,
                "fused_max_cos": fused_max_cos,
                "fused_mean_cos": fused_mean_cos,
                "rank1_baseline": rank1_base,
                "rank5_baseline": rank5_base,
                "rank5_correct": rank5_gen,
                "rank1_fused_max": rank1_fmax,
                "rank1_fused_mean": rank1_fmean,
                "rank1_gen_leaky": rank1_gen_leaky,
                "generated_path": pair_name,
            })
            f.flush()

            running_id_scores.append(id_score)
            running_age_maes.append(age_mae)
            n_success += 1
            status = (f"OK (Fallback: {preprocessing_fallback}) | gen={id_score:.3f} "
                      f"gốc={baseline_cos:.3f} | R1 gen/gốc={rank1_correct_fgnet}/{rank1_base}")

            # ====================================================================
            # 6. HIỂN THỊ INLINE 4 CỘT TRỰC QUAN
            # Source gốc -> Source đã tiền xử lý -> Generated -> Target thật
            # ====================================================================
            if SHOW_INLINE_IMAGES and (idx % INLINE_DISPLAY_FREQ == 0 or idx == len(remaining_pairs)):
                try:
                    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
                    
                    orig_disp = cv2.cvtColor(cv2.imread(pair["source_img"]), cv2.COLOR_BGR2RGB)
                    axes[0].imshow(orig_disp)
                    axes[0].set_title(f"1. Source gốc ({pair['source_age']}t)\n{os.path.basename(pair['source_img'])}", fontsize=10)
                    axes[0].axis("off")

                    prep_disp = preprocessed_img_for_display if preprocessed_img_for_display is not None else orig_disp
                    axes[1].imshow(prep_disp)
                    prep_status = "[FALLBACK GỐC]" if preprocessing_fallback else "[Pad + SoG WB + CF w=0.7]"
                    axes[1].set_title(f"2. Source Tiền Xử Lý\n{prep_status}", fontsize=10, color="orange" if preprocessing_fallback else "green", fontweight="bold")
                    axes[1].axis("off")

                    axes[2].imshow(Image.open(generated))
                    axes[2].set_title(f"3. Generated ({pair['target_age']}t)\nID: {id_score:.3f} | Tuổi: {estimated_age:.1f}", fontsize=10, color="blue", fontweight="bold")
                    axes[2].axis("off")

                    tgt_disp = cv2.cvtColor(cv2.imread(pair["target_img"]), cv2.COLOR_BGR2RGB)
                    axes[3].imshow(tgt_disp)
                    axes[3].set_title(f"4. Target Thật ({pair['target_age']}t)\nGround Truth ({os.path.basename(pair['target_img'])})", fontsize=10)
                    axes[3].axis("off")

                    plt.tight_layout()
                    plt.show()
                except Exception as e_plot:
                    print(f"Lỗi hiển thị ảnh inline: {e_plot}")

        except Exception as e:
            n_failed += 1
            status = f"LỖI: {e}"

        elapsed = time.time() - t_start
        avg_per_pair = elapsed / idx
        eta = avg_per_pair * (len(remaining_pairs) - idx)
        pair_time = time.time() - t_pair_start

        print(f"[{idx}/{len(remaining_pairs)}] người={pair['person_id']} "
              f"{pair['source_age']}->{pair['target_age']} tuổi | {pair_time:.1f}s | {status}")

        if idx % 10 == 0 or idx == len(remaining_pairs):
            avg_id = sum(running_id_scores) / len(running_id_scores) if running_id_scores else float("nan")
            avg_mae = sum(running_age_maes) / len(running_age_maes) if running_age_maes else float("nan")
            print("-" * 75)
            print(f"  TỔNG KẾT sau {idx} cặp: thành công={n_success}, lỗi={n_failed} (Fallback tiền xử lý: {n_fallback})")
            print(f"  ID Score TB={avg_id:.4f} | Age MAE TB={avg_mae:.2f} năm")
            print(f"  Đã chạy: {timedelta(seconds=int(elapsed))} | Còn lại (ước tính): {timedelta(seconds=int(eta))}")
            print("-" * 75)

print("\n" + "=" * 75)
print(f"HOÀN TẤT ĐÁNH GIÁ FG-NET.")
print(f"Tổng: thành công={n_success}, lỗi={n_failed}, số cặp fallback tiền xử lý={n_fallback}")
print(f"Kết quả đã lưu tại: {RESULTS_CSV}")
print("=" * 75)


In [ ]:
import pandas as pd

df_results = pd.read_csv(RESULTS_CSV)
print(f"Tổng số cặp đã có kết quả (cộng dồn các phiên): {len(df_results)}")
print()
print(f"🎯 ID Score trung bình TOÀN BỘ (Baseline Mới): {df_results['id_score'].mean():.4f}")
print(f"🎯 Age MAE trung bình TOÀN BỘ: {df_results['age_mae'].mean():.2f} năm")

if "preprocessing_fallback" in df_results.columns:
    n_fallback = df_results["preprocessing_fallback"].sum()
    pct_fallback = df_results["preprocessing_fallback"].mean() * 100
    print(f"\n--- Thống kê Fallback Tiền xử lý ---")
    print(f"Số cặp bị fallback về ảnh gốc: {n_fallback}/{len(df_results)} ({pct_fallback:.1f}%)")
    print(f"  -> Đối chiếu trực tiếp với con số ~8.5% lỗi kỹ thuật cũ khi chưa có tiền xử lý.")
    if n_fallback > 0 and len(df_results) > n_fallback:
        succ_mask = ~df_results["preprocessing_fallback"]
        fb_mask = df_results["preprocessing_fallback"]
        print(f"  -> ID Score TB cặp Tiền xử lý thành công : {df_results[succ_mask]['id_score'].mean():.4f}")
        print(f"  -> ID Score TB cặp bị Fallback về gốc    : {df_results[fb_mask]['id_score'].mean():.4f}")

if "gender_match" in df_results.columns:
    acc = df_results["gender_match"].mean() * 100
    print(f"\nGender Match Rate (Độ chính xác giới tính): {acc:.2f}% ({df_results['gender_match'].sum()}/{len(df_results)})")
if "estimated_gender" in df_results.columns:
    print(f"Phân bổ giới tính dự đoán (estimated_gender): {df_results['estimated_gender'].value_counts().to_dict()}")

print("\n--- Theo khoảng cách tuổi (vấn đề #5) ---")
df_results["age_gap"] = (df_results["target_age"] - df_results["source_age"]).abs()
df_results["gap_bucket"] = pd.cut(df_results["age_gap"], bins=[0,2,4,6,8,10,15,20,30,50,100])
print(df_results.groupby("gap_bucket", observed=True)[["id_score", "age_mae"]].mean())

print("\n--- Riêng nhóm source_age < 15 (vấn đề #1, trẻ em) ---")
child_mask = df_results["source_age"] < 15
print(f"Số cặp trẻ em (<15t): {child_mask.sum()}")
if child_mask.sum() > 0:
    print(f"ID Score TB (trẻ em): {df_results[child_mask]['id_score'].mean():.4f}")
    print(f"ID Score TB (còn lại): {df_results[~child_mask]['id_score'].mean():.4f}")


### v2 — Ảnh sinh có giúp tìm người tốt hơn ảnh gốc không?

So sánh **ghép cặp** trên cùng các cặp và cùng embedder: truy vấn bằng ảnh nguồn gốc (baseline), bằng ảnh sinh (FADING), và kết hợp hai vector. Rank-k tính trên gallery **đã loại chính ảnh nguồn** — bản cũ giữ ảnh nguồn trong gallery nên Rank-1 của ảnh sinh bị thổi phồng (ảnh sinh dựng từ ảnh nguồn).


In [ ]:
# ===== v2: Ảnh sinh vs ảnh gốc vs kết hợp — so sánh GHÉP CẶP trên cùng pipeline =====
from scipy.stats import wilcoxon

df_v2 = pd.read_csv(RESULTS_CSV)
if "baseline_cos" not in df_v2.columns:
    print("CSV chưa có cột v2 — chạy lại cell đánh giá v2.")
else:
    def _row(label, d):
        if len(d) == 0:
            return None
        delta = d["id_score"] - d["baseline_cos"]
        return {
            "nhóm": label, "n": len(d),
            "cos gốc→đích": round(d["baseline_cos"].mean(), 4),
            "cos sinh→đích": round(d["id_score"].mean(), 4),
            "cos kết hợp (mean)": round(d["fused_mean_cos"].mean(), 4),
            "sinh thắng (%)": round((delta > 0).mean() * 100, 1),
            "p Wilcoxon": float(f"{wilcoxon(d['id_score'], d['baseline_cos']).pvalue:.3g}") if len(d) > 5 else None,
            "R1 gốc (%)": round(d["rank1_baseline"].mean() * 100, 1),
            "R1 sinh (%)": round(d["rank1_correct"].mean() * 100, 1),
            "R1 kết hợp max (%)": round(d["rank1_fused_max"].mean() * 100, 1),
            "R1 kết hợp mean (%)": round(d["rank1_fused_mean"].mean() * 100, 1),
            "R5 gốc (%)": round(d["rank5_baseline"].mean() * 100, 1),
            "R5 sinh (%)": round(d["rank5_correct"].mean() * 100, 1),
            "R1 sinh CŨ-rò rỉ (%)": round(d["rank1_gen_leaky"].mean() * 100, 1),
        }

    gap = df_v2["target_age"] - df_v2["source_age"]
    rows = [r for r in [
        _row("Tất cả", df_v2),
        _row("Nguồn < 15 tuổi", df_v2[df_v2["source_age"] < 15]),
        _row("Nguồn ≥ 15 tuổi", df_v2[df_v2["source_age"] >= 15]),
        _row("Cách tuổi ≤ 10", df_v2[gap <= 10]),
        _row("Cách tuổi > 10", df_v2[gap > 10]),
    ] if r]
    df_cmp = pd.DataFrame(rows)
    METRICS_REPORT_DIR = os.path.join(OUTPUT_DIR, "metrics_report")
    os.makedirs(METRICS_REPORT_DIR, exist_ok=True)
    df_cmp.to_csv(os.path.join(METRICS_REPORT_DIR, "generative_vs_source_v2.csv"), index=False)
    display(df_cmp)
    print("Rank-k tính trên gallery ĐÃ LOẠI ảnh nguồn. Cột 'CŨ-rò rỉ' chỉ để đối chiếu với báo cáo trước.")


### Chất lượng sinh ảnh theo ĐỘ TUỔI ĐÍCH (target_age) — xác định vùng tuổi hệ thống sinh tốt/hạn chế

Khác với phân tích theo khoảng cách tuổi (age_gap) hay tuổi nguồn (trẻ em) — đây trả lời trực tiếp: **"sinh RA đúng độ tuổi nào thì tốt, độ tuổi nào thì kém"**. Tự động lưu ra CSV + biểu đồ, không cần chạy lại pipeline.

In [ ]:
import matplotlib.pyplot as plt

METRICS_REPORT_DIR = os.path.join(OUTPUT_DIR, "metrics_report")
os.makedirs(METRICS_REPORT_DIR, exist_ok=True)

target_bins = [0, 5, 10, 15, 20, 30, 40, 50, 60, 100]
df_results["target_age_bucket"] = pd.cut(df_results["target_age"], bins=target_bins)

target_summary = df_results.groupby("target_age_bucket", observed=True)["id_score"].agg(["mean", "count", "std"])
print("=== Chất lượng sinh ảnh theo TỪNG KHOẢNG TUỔI ĐÍCH (FG-NET) ===")
print(target_summary)

TARGET_AGE_SUMMARY_CSV = os.path.join(METRICS_REPORT_DIR, "target_age_quality_summary.csv")
target_summary.to_csv(TARGET_AGE_SUMMARY_CSV)
print(f"\nĐã lưu: {TARGET_AGE_SUMMARY_CSV}")

fig, ax = plt.subplots(figsize=(10, 5))
x_labels = [str(b) for b in target_summary.index]
ax.plot(x_labels, target_summary["mean"], marker="o", color="#7c3aed", linewidth=2)
ax.axhline(0.6, color="red", linestyle="--", alpha=0.6, label="Ngưỡng ArcFace (0.6)")
ax.set_title("ID Score theo ĐỘ TUỔI ĐÍCH (target_age) — FG-NET")
ax.set_xlabel("Khoảng tuổi đích (năm)")
ax.set_ylabel("ID Score trung bình")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=30)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

TARGET_AGE_CHART_PATH = os.path.join(METRICS_REPORT_DIR, "id_score_vs_target_age.png")
plt.savefig(TARGET_AGE_CHART_PATH, dpi=120)
plt.show()
print(f"Đã lưu biểu đồ: {TARGET_AGE_CHART_PATH}")

overall_mean = df_results["id_score"].mean()
above_avg = target_summary[target_summary["mean"] > overall_mean]
print(f"\nID Score trung bình CHUNG: {overall_mean:.4f}")
if len(above_avg) > 0:
    print("Các khoảng tuổi đích có ID Score TRÊN trung bình chung:")
    print(above_avg)


### Vẽ đường cong + tìm ngưỡng gãy

Chỉ đọc lại `fgnet_eval_results.csv` đã có (không chạy lại pipeline) — nếu Mục 1 (đo ID Score/MAE) chưa chạy xong hoặc `df_results` rỗng, chạy lại cell "Tổng hợp cuối cùng" ở trên trước.

In [ ]:
import matplotlib.pyplot as plt

ARCFACE_THRESHOLD = 0.6
gap_summary = df_results.groupby("gap_bucket", observed=True)["id_score"].mean()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(gap_summary.index.astype(str), gap_summary.values, marker="o", color="#2563eb", linewidth=2)
plt.xticks(rotation=30)
ax1.set_xlabel("Khoảng cách tuổi (năm)")
ax1.set_ylabel("ID Score trung bình")
ax1.set_title("ID Score theo khoảng cách tuổi - FG-NET")
ax1.axhline(y=ARCFACE_THRESHOLD, color="red", linestyle="--", alpha=0.5, label=f"Ngưỡng ArcFace ({ARCFACE_THRESHOLD})")
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(METRICS_REPORT_DIR, "id_score_vs_age_gap.png")
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Đã lưu biểu đồ tại: {fig_path}")

gap_diffs = gap_summary.diff().abs()
print("\nĐộ giảm giữa mỗi bước liên tiếp:")
print(gap_diffs)

if gap_diffs.notna().sum() >= 2:
    threshold_step = gap_diffs.idxmax()
    other_diffs = gap_diffs.drop(threshold_step).dropna()
    print(f"\n>>> Ngưỡng gãy phát hiện: tại khoảng '{threshold_step}'")
    print(f"    Độ giảm tại đây: {gap_diffs.max():.4f}")


### Đo riêng nhóm tuổi trẻ em — đối chiếu phát hiện của DiffAge3D

Chỉ đọc lại `df_results` đã có (không chạy lại pipeline). DiffAge3D chỉ ra rằng FADING gặp hạn chế khi biến đổi ảnh trẻ mới biết đi (toddler, ~0-5 tuổi) sang người trung niên — do cơ chế Attention Control giữ nguyên cấu trúc không gian khuôn mặt, trong khi trẻ em → người lớn cần thay đổi cả cấu trúc xương mặt, không chỉ texture bề mặt.

In [ ]:
# ----- Viec 1: so sanh nhanh, 2 nhom (tre em vs con lai) -----
child_mask = df_results["source_age"] < 15
n_child = child_mask.sum()
n_rest = (~child_mask).sum()

print("=== Viec 1: So sanh tong quat (nguong < 15 tuoi) ===")
print(f"So cap nhom tre em (source_age < 15): {n_child}")
print(f"So cap nhom con lai: {n_rest}")

if n_child > 0:
    id_child = df_results[child_mask]["id_score"].mean()
    id_rest = df_results[~child_mask]["id_score"].mean()
    print(f"ID Score TB (tre em):  {id_child:.4f}")
    print(f"ID Score TB (con lai): {id_rest:.4f}")
    print(f"Chenh lech: {id_rest - id_child:+.4f} ({(id_rest - id_child) / id_rest * 100:+.1f}%)")
else:
    print("CHUA CO du du lieu nhom tre em - can chay them cap trong vong lap chinh o tren.")

print()

# ----- Viec 2: chia nho hon, dung theo dinh nghia "toddler" cua DiffAge3D -----
print("=== Viec 2: Chia nho theo dai tuoi (khop dinh nghia 'toddler' 0-5 tuoi) ===")
bins = [0, 5, 10, 15, 100]
labels = ["0-5 (toddler)", "6-10", "11-14", "15+"]
df_results["age_group_fine"] = pd.cut(df_results["source_age"], bins=bins, labels=labels)

fine_summary = df_results.groupby("age_group_fine", observed=True).agg(
    id_score_mean=("id_score", "mean"),
    age_mae_mean=("age_mae", "mean"),
    n_cap=("id_score", "count"),
)
print(fine_summary)

# ----- Ket luan tu dong, dua tren so lieu that -----
print()
if "0-5 (toddler)" in fine_summary.index and len(fine_summary) > 1:
    toddler_score = fine_summary.loc["0-5 (toddler)", "id_score_mean"]
    other_scores = fine_summary.drop("0-5 (toddler)")["id_score_mean"]
    if toddler_score < other_scores.min():
        print(">>> KET LUAN: nhom 0-5 tuoi (toddler) co ID Score THAP NHAT trong tat ca cac nhom,")
        print("    NHAT QUAN voi phat hien cua DiffAge3D ve han che cua FADING voi anh toddler.")
    else:
        print(">>> KET LUAN: nhom 0-5 tuoi KHONG phai nhom thap nhat trong du lieu nay -")
        print("    can xem lai so luong mau (co the qua it de ket luan chac chan).")
else:
    print("Chua du du lieu nhom '0-5 (toddler)' de dua ra ket luan.")


## Báo cáo Metrics Tổng hợp Chuyên biệt — FG-NET
Báo cáo toàn diện các chỉ số: Mean/Median/Std, Khoảng tin cậy 95% (CI 95%), Tỷ lệ chấp nhận (≥60%),
Rank-1 Accuracy trong Gallery, Tỷ lệ lỗi kỹ thuật, Runtime/Thông lượng và Tỷ lệ Fallback Tiền xử lý.


In [ ]:
from scipy import stats
import pandas as pd

METRICS_REPORT_DIR = os.path.join(OUTPUT_DIR, "metrics_report")
os.makedirs(METRICS_REPORT_DIR, exist_ok=True)

df_fgnet_final = pd.read_csv(RESULTS_CSV)
ACCEPT_THRESHOLD = 0.6  # Ngưỡng chấp nhận chuẩn

def read_total_pairs(txt_path: str, fallback_len: int) -> int:
    """Đọc tổng số cặp dự kiến từ file lưu lúc tạo cặp để tính đúng tỷ lệ lỗi khi resume."""
    if os.path.exists(txt_path):
        with open(txt_path) as f:
            return int(f.read().strip())
    return fallback_len

def summarize_table(df: pd.DataFrame, score_col: str, label: str, total_pairs: int) -> dict:
    scores = df[score_col].dropna()
    n = len(scores)
    mean = scores.mean()
    std = scores.std()

    if n >= 2:
        ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=std / (n ** 0.5))
    else:
        ci_low, ci_high = (float("nan"), float("nan"))

    n_failed = max(total_pairs - len(df), 0)

    row = {
        "Bảng": label,
        "N thành công": n,
        "N dự kiến": total_pairs,
        "Tỷ lệ lỗi kỹ thuật (%)": round(n_failed / total_pairs * 100, 2) if total_pairs else float("nan"),
        "Mean Score": round(mean, 4),
        "Median Score": round(scores.median(), 4),
        "Std Score": round(std, 4),
        "CI 95% thấp": round(ci_low, 4),
        "CI 95% cao": round(ci_high, 4),
        f"Tỷ lệ chấp nhận (≥{int(ACCEPT_THRESHOLD*100)}%)": round((scores >= ACCEPT_THRESHOLD).mean() * 100, 2),
        "Age MAE TB (năm)": round(df["age_mae"].mean(), 2),
    }
    if "rank1_correct" in df.columns:
        row["Rank-1 Accuracy (%)"] = round(df["rank1_correct"].mean() * 100, 2)
    if "runtime_sec" in df.columns:
        mean_runtime = df["runtime_sec"].mean()
        row["Runtime TB (giây/cặp)"] = round(mean_runtime, 1)
        row["Thông lượng ước tính (ca/giờ)"] = round(3600 / mean_runtime, 1) if mean_runtime else float("nan")
    if "preprocessing_fallback" in df.columns:
        row["Tỷ lệ Fallback Tiền xử lý (%)"] = round(df["preprocessing_fallback"].mean() * 100, 2)
    return row

fgnet_total = read_total_pairs(os.path.join(OUTPUT_DIR, "fgnet_total_pairs.txt"), len(df_fgnet_final))

rows = []
rows.append(summarize_table(
    df_fgnet_final, "id_score",
    "FG-NET — Cross-age Re-identification (vs ảnh thật ở tuổi đích)",
    fgnet_total,
))

# Phân nhóm trẻ em / còn lại
fg_child = df_fgnet_final[df_fgnet_final["source_age"] < 15]
fg_rest = df_fgnet_final[df_fgnet_final["source_age"] >= 15]

if len(fg_child) > 0:
    rows.append(summarize_table(fg_child, "id_score", "  ↳ FG-NET, nguồn < 15 tuổi", len(fg_child)))
if len(fg_rest) > 0:
    rows.append(summarize_table(fg_rest, "id_score", "  ↳ FG-NET, nguồn ≥ 15 tuổi", len(fg_rest)))

summary_report_df = pd.DataFrame(rows)
print(summary_report_df.to_string(index=False))

summary_report_df.to_csv(os.path.join(METRICS_REPORT_DIR, "summary_fgnet.csv"), index=False)

readme_text = f"""# Báo cáo Đánh giá Chuyên biệt — FG-NET Dataset

## Bản chất đánh giá: Cross-age Re-identification
- id_score so sánh ảnh sinh ra với ẢNH THẬT KHÁC của cùng người ở đúng tuổi đích (ground truth).
- Phản ánh chính xác năng lực nhận diện người mất tích qua các mốc thời gian thực tế.

## Thống kê tổng hợp
- N thành công: {len(df_fgnet_final)} cặp.
- Mean ID Score: {df_fgnet_final['id_score'].mean():.4f} (ngưỡng chấp nhận >= 0.6).
- Age MAE TB: {df_fgnet_final['age_mae'].mean():.2f} năm.
- Rank-1 Accuracy: {df_fgnet_final['rank1_correct'].mean()*100:.2f}%.
"""
with open(os.path.join(METRICS_REPORT_DIR, "README_giai_thich_metrics_fgnet.md"), "w", encoding="utf-8") as f:
    f.write(readme_text)

print(f"\n✅ Đã lưu báo cáo tổng hợp FG-NET vào: {METRICS_REPORT_DIR}/summary_fgnet.csv")


## Đóng gói Toàn bộ Kết quả FG-NET & Tạo Link Tải về
Nén toàn bộ file kết quả đánh giá `fgnet_eval_results.csv`, ảnh sinh ra, ảnh đã căn chỉnh và thư mục `metrics_report/` thành 1 file zip duy nhất `fgnet_results_and_metrics.zip`.


In [ ]:
import zipfile
import subprocess
from IPython.display import FileLink, display

ZIP_FGNET_PATH = "/kaggle/working/fgnet_results_and_metrics.zip"
n_total = 0

with zipfile.ZipFile(ZIP_FGNET_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # 1. File kết quả CSV chính
    if os.path.isfile(RESULTS_CSV):
        zf.write(RESULTS_CSV, arcname=os.path.basename(RESULTS_CSV))
        n_total += 1

    # 2. Thư mục báo cáo metrics (CSV, biểu đồ, README)
    metrics_dir = os.path.join(OUTPUT_DIR, "metrics_report")
    if os.path.isdir(metrics_dir):
        for root, _, files in os.walk(metrics_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                arcname = os.path.join("metrics_report", os.path.relpath(fpath, metrics_dir))
                zf.write(fpath, arcname=arcname)
                n_total += 1

    # 3. Thư mục ảnh sinh ra & căn chỉnh
    for folder_name in ["fgnet_generated_pairs", "fgnet_embeddings", "fgnet_aligned_inputs",
                        "fgnet_preprocessed_sources"]:
        folder_path = os.path.join(OUTPUT_DIR, folder_name)
        if os.path.isdir(folder_path):
            for root, _, files in os.walk(folder_path):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    arcname = os.path.join(folder_name, os.path.relpath(fpath, folder_path))
                    zf.write(fpath, arcname=arcname)
                    n_total += 1

zip_size = os.path.getsize(ZIP_FGNET_PATH) / (1024 * 1024) if os.path.isfile(ZIP_FGNET_PATH) else 0
print(f"\n✅ Đã tạo gói nén kết quả FG-NET: {ZIP_FGNET_PATH} ({zip_size:.1f} MB, {n_total} files)")
print("   Bao gồm: CSV v2, ảnh sinh theo cặp, embedding .npy/.npz, aligned, metrics_report/")
display(FileLink("fgnet_results_and_metrics.zip"))
